<p class="lead">Between downloading a dataset and fitting a model lies a stage no paper describes and every project spends most of its time in: turning records as a database delivered them into a table whose rows, columns and units each mean exactly one thing. We take the laboratory's running dataset — 5,089 bioactivity measurements for the kinase ROCK1, exactly as the ChEMBL web API returned them — and work through the repertoire of <strong>dplyr</strong> and <strong>tidyr</strong> needed to reshape, join, aggregate, de-duplicate and unit-harmonise it into a table with one row per compound and one trustworthy number per row. Every operation comes with its assumptions, its cost and at least one way of getting it silently wrong. This is the R edition of the unit; the Python edition does the same work with pandas.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad enseña a transformar una descarga cruda en una tabla modelable usando el <em>tidyverse</em>. Partimos de los <em>datos ordenados</em> (una variable por columna, una observación por fila, una unidad experimental por tabla) y recorremos el repertorio de manipulación: remodelado (<code>pivot_longer</code>, <code>pivot_wider</code>, <code>nest</code>/<code>unnest</code>, <code>separate_rows</code>); uniones seguras con <code>relationship=</code> y <code>anti_join</code>; <code>group_by</code> con <code>summarise</code> y <code>mutate</code>; los mecanismos de datos faltantes (MCAR, MAR, MNAR); duplicados por clave, por bandera y por estructura (InChIKey); armonización de unidades; valores censurados y la inversión del símbolo al tomar logaritmos; texto, factores y fechas; el operador <code>|&gt;</code>; y rendimiento (<code>data.table</code> frente a <code>dplyr</code>). El producto es la tabla a nivel de compuesto con sus invariantes verificadas mediante <code>stopifnot()</code>.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>State the tidy-data rules and reshape any table into tidy form with <code>pivot_longer</code>, <code>pivot_wider</code>, <code>nest</code>/<code>unnest</code> and <code>separate_rows</code>.</li>
<li>Combine tables correctly: choose the join verb, declare the expected cardinality with <code>relationship=</code>, audit the match with <code>anti_join</code>, and recognise the many-to-many explosion.</li>
<li>Aggregate with <code>group_by</code> + <code>summarise</code> and normalise within groups with <code>group_by</code> + <code>mutate</code> and window functions.</li>
<li>Distinguish MCAR, MAR and MNAR missingness, diagnose which one a column has, and choose a defensible response.</li>
<li>Remove duplicates deliberately: by exact key, by database flag, and by standardised chemical structure (InChIKey).</li>
<li>Harmonise mixed units with an explicit conversion table, convert to a logarithmic activity scale, and handle censored (<code>&gt;</code>, <code>&lt;</code>) measurements without discarding them.</li>
<li>Clean strings with regular expressions in <code>stringr</code>, use factors and date accessors, and write wrangling code as a readable pipeline.</li>
<li>Produce the compound-level modelling table from measurement-level records and assert its invariants in code.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read it linearly, running each chunk and reading its output; the prose after every chunk says what to look for. Green <strong>domain-context boxes</strong> define each chemical or biological term on first use. Attempt the exercises before reading on. In a fresh Google Colab R runtime, run once in a cell of your own: <code>install.packages(c("tidyverse","data.table","microbenchmark","patchwork"))</code>. This unit follows Unit 3 and feeds Units 5, 6 and 11–12.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(stringr); library(lubridate)
  library(ggplot2); library(patchwork); library(data.table); library(microbenchmark)
})
set.seed(0)                       # every random draw in this document is reproducible
options(width = 120, digits = 6)
invisible(Sys.setlocale("LC_CTYPE", "en_US.UTF-8"))   # so that 'uM' and Greek mu survive

PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 10) +
          theme(plot.title = element_text(face = "bold", size = 10.5),
                panel.grid.minor = element_blank(),
                axis.line = element_line(colour = "grey40", linewidth = 0.3)))

REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO,
                   "/main/DataScience/data/")
data_path <- function(name) {
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)                       # Colab: download once
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE)
  }
  dest
}
load_data <- function(name, ...)
  read.csv(data_path(name), stringsAsFactors = FALSE, na.strings = c("", "NA"), ...)

cat("R", as.character(getRversion()), "| dplyr", as.character(packageVersion("dplyr")),
    "| tidyr", as.character(packageVersion("tidyr")),
    "| data.table", as.character(packageVersion("data.table")),
    "| ggplot2", as.character(packageVersion("ggplot2")), "\n")
cat("data folder:", dirname(data_path("rock1_chembl_raw.csv")),
    "| raw file found:", file.exists(data_path("rock1_chembl_raw.csv")), "\n")



The versions are printed so that a reader who gets a different number three years from now knows where to look. `set.seed(0)` fixes the random-number stream; `stringsAsFactors = FALSE` and `na.strings = c("", "NA")` are the two arguments that decide how `read.csv` interprets a bioactivity file, and both are set explicitly rather than inherited from a default that changed between R versions.

## 1. Tidy data: one rule you will use every day

A table can store the same information in many shapes, and most shapes make analysis harder than it needs to be. Hadley Wickham's **tidy-data** formulation<sup class="c"><a href="#r1">[1]</a></sup> reduces the choice to three rules:

1. **Each variable forms a column.** A *variable* is something that was measured or that labels a measurement: the compound, the assay, the potency, the unit.
2. **Each observation forms a row.** An *observation* is one application of the measuring instrument — here, one compound tested once in one assay.
3. **Each type of observational unit forms a table.** Measurements belong in one table; the properties of the assays those measurements came from belong in another, joined by a key.

The rules are a contract with the rest of your toolbox: `group_by` expects the grouping variable in a column, `ggplot2` expects the aesthetic mapping in columns, every modelling function expects one row per example. Untidy data can always be analysed — with enough bespoke code — but every bespoke line is a place for a bug to live.<sup class="c"><a href="#r2">[2]</a></sup> These three rules are the design principle of the R *tidyverse*<sup class="c"><a href="#r27">[27]</a></sup> and, in Python, of the pandas library that borrowed them.<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>

The most common violation in science is the **wide** table: a spreadsheet in which the column *headers are values*, not variable names. A biologist records three concentrations as three columns; a chemist records IC50, Ki and Kd as three columns. The information is all there, but "which measurement type" has been smuggled into the header row, where no software can reach it. The operation that rescues such a table is `pivot_longer` (historically *melt* or *gather*): it turns a block of columns into two columns, one holding the former header and one holding the value. Its inverse, `pivot_wider`, goes back.

The chunk below builds a deliberately untidy miniature of our real problem — four compounds, three measurement types, values in the cells — and tidies it.



In [ ]:
wide <- tibble::tibble(
  compound = c("fasudil", "Y-27632", "ripasudil", "belumosudil"),
  IC50_nM  = c(330, 150, 51, 7400),
  Ki_nM    = c(330, 140, NA, 9900),
  Kd_nM    = c(NA, 220, NA, NA))
cat("UNTIDY (wide): 'IC50_nM', 'Ki_nM', 'Kd_nM' are values of one variable, hidden in the header\n")
print(as.data.frame(wide))

tidy <- wide |>
  pivot_longer(-compound, names_to = "measure", values_to = "value_nM") |>
  mutate(measure = str_remove(measure, "_nM")) |>
  filter(!is.na(value_nM)) |>
  arrange(compound, measure)
cat("\nTIDY (long): one row per observation, one column per variable\n")
print(as.data.frame(tidy))
cat("\nrows: wide =", nrow(wide), "| tidy =", nrow(tidy),
    "(the NA cells were never observations)\n")



Read the two printouts side by side. The wide table has 4 rows and 12 cells, of which 4 are empty: those empty cells are not missing data, they are *measurements that were never made*, and the wide shape forces us to represent them anyway. The tidy table has 8 rows, one per measurement that actually exists, and the fact that ripasudil has only an IC50 is expressed by its having only one row.

Note the shape of the code as much as its result: `pivot_longer(-compound, ...)` uses tidy-select ("every column except `compound`"), and `filter(!is.na(value_nM))` discards the cells that never were observations deliberately, where `values_drop_na = TRUE` would do it invisibly.

Two warnings. Tidy is a property of a table *with respect to a question*: the compound-level table built at the end of this unit is tidy for modelling, the measurement-level table is tidy for assessing reproducibility, and they are different tables for different observational units — rule 3, not a contradiction. And tidiness is not convenience: a 2,048-column fingerprint matrix is the right shape for a model matrix.

<div class="exercise"><h4>Exercise 1</h4>
<p>Transpose <code>wide</code> so the four compounds are columns and the three measurement types are rows (<code>pivot_longer</code> then <code>pivot_wider</code>), without losing the missing-value pattern, then recover <code>tidy</code> from it. Which tidy rule does the transposed table break, and what happens to the column types if you use <code>t()</code> from base R instead?</p></div>

The conceptual figure below fixes the vocabulary we will use for the rest of the course.



In [ ]:
cells_untidy <- tibble::tribble(
  ~row, ~col, ~txt,
  1, 1, "compound", 1, 2, "IC50_nM", 1, 3, "Ki_nM",
  2, 1, "fasudil",  2, 2, "330",     2, 3, "330",
  3, 1, "Y-27632",  3, 2, "150",     3, 3, "140",
  4, 1, "ripasudil",4, 2, "51",      4, 3, "\u2014")
cells_tidy <- tibble::tribble(
  ~row, ~col, ~txt,
  1, 1, "compound", 1, 2, "measure", 1, 3, "value_nM",
  2, 1, "fasudil",  2, 2, "IC50",    2, 3, "330",
  3, 1, "fasudil",  3, 2, "Ki",      3, 3, "330",
  4, 1, "Y-27632",  4, 2, "IC50",    4, 3, "150",
  5, 1, "Y-27632",  5, 2, "Ki",      5, 3, "140",
  6, 1, "ripasudil",6, 2, "IC50",    6, 3, "51")
draw_table <- function(d, title, head_fill) {
  d <- mutate(d, head = row == 1)
  ggplot(d, aes(col, -row)) +
    geom_tile(aes(fill = head), colour = "grey35", linewidth = 0.3, width = 0.98, height = 0.9) +
    geom_text(aes(label = txt, fontface = ifelse(head, "bold", "plain")), size = 2.7) +
    scale_fill_manual(values = c(`TRUE` = head_fill, `FALSE` = "grey97"), guide = "none") +
    labs(title = title) + theme_void(base_size = 9) +
    theme(plot.title = element_text(face = "bold", size = 9, hjust = 0.5))
}
p1 <- draw_table(cells_untidy, "untidy: headers are values\n(the variable 'measure' has no column)", "#e8b7a0")
p2 <- draw_table(cells_tidy, "tidy: one variable per column,\none observation per row", "#b9cce4")
print(p1 | p2)



<p class="caption"><b>Figure 1.</b> Two shapes of the same five measurements. Left, untidy: the shaded header cells <code>IC50_nM</code> and <code>Ki_nM</code> are <em>values</em> of a variable that has no column, and the em-dash marks a measurement never made but represented anyway. Right, the tidy table produced by <code>pivot_longer</code>: six rows (one header, five measurements) for the five measurements that exist, three columns for the three variables.</p>

## 2. The raw table, exactly as it arrived

The running dataset of this course is a **frozen snapshot of ChEMBL release 37** (released 2026-05-01, downloaded 2026-10-18) holding every IC50, Ki and Kd measurement for the human protein ROCK1, target `CHEMBL3231`.<sup class="c"><a href="#r3">[3]</a></sup><sup class="c"><a href="#r4">[4]</a></sup> It is frozen deliberately: a live API returns a different table every month, and a notebook whose numbers change is not reproducible.<sup class="c"><a href="#r5">[5]</a></sup> Nothing was cleaned before freezing — the defects are the lesson. The R and Python editions of this course read exactly the same file, so the two can be compared line by line.

<div class="box bridge"><h4>Domain context — what a bioactivity measurement is</h4>
<p>A <strong>protein kinase</strong> is an enzyme that attaches a phosphate group to other proteins, switching them on or off; ROCK1 (Rho-associated protein kinase 1) is one such enzyme and one of this laboratory's targets. A drug-discovery project looks for small molecules — <strong>compounds</strong>, a few dozen atoms each — that block it, and "blocking" is quantified by a concentration: the <strong>IC50</strong> halves the enzyme's activity in a given experiment, the <strong>Ki</strong> is the thermodynamic binding constant, the <strong>Kd</strong> its equivalent for binding without catalysis. All three are concentrations, usually in nanomolar (nM, 10<sup>−9</sup> mol/L), and all three are <em>smaller is better</em>: IC50 = 5 nM is a hundred times more potent than 500 nM. An <strong>assay</strong> is one experimental protocol (enzyme preparation, substrate, detection chemistry); the same compound in two assays can differ by an order of magnitude, which is why the assay identifier travels with every measurement. <strong>ChEMBL</strong> is the public database, curated at the European Bioinformatics Institute, that extracts such measurements from the medicinal-chemistry literature and patents, and increasingly accepts direct deposition.<sup class="c"><a href="#r3">[3]</a></sup><sup class="c"><a href="#r32">[32]</a></sup> IC50 and Ki are related by the Cheng–Prusoff equation when the assay conditions are known,<sup class="c"><a href="#r6">[6]</a></sup> which they usually are not.</p></div>

First contact with a new table asks four questions: how big, what types, how much memory, what does a row look like?



In [ ]:
raw <- load_data("rock1_chembl_raw.csv")
cat("dimensions:", nrow(raw), "rows x", ncol(raw), "columns\n")
cat("memory:", sprintf("%.1f MB", as.numeric(object.size(raw)) / 1e6), "\n\n")
cat("columns and classes:\n")
print(data.frame(class = sapply(raw, class)))
cat("\none row, transposed so every field is readable:\n")
print(t(raw[1, c("activity_id", "molecule_chembl_id", "type", "relation", "value", "units",
                 "standard_type", "standard_relation", "standard_value", "standard_units",
                 "pchembl_value", "assay_chembl_id", "assay_type", "document_chembl_id",
                 "target_organism", "potential_duplicate")]))



Look at the class list: `read.csv` with `stringsAsFactors = FALSE` reads text as `character` and numbers as `integer` or `numeric`, and has already made one decision for us — `max_phase` is `numeric` because it holds missing values alongside fractional phases. Left at its pre-4.0 default, `stringsAsFactors` would have turned every identifier column into a factor whose levels we would fight for the rest of the unit.

Read the transposed first row as a sentence: *activity record `activity_id` says that molecule `molecule_chembl_id` gave an `IC50` of `value` `units` in assay `assay_chembl_id`, reported in document `document_chembl_id`.* Notice that the measurement appears **twice**: as `type`/`relation`/`value`/`units` (as the paper printed them) and as `standard_type`/`standard_relation`/`standard_value`/`standard_units` (ChEMBL's harmonisation). The pair is a gift — it lets us build our own harmonisation and check it against a curated one, which Section 9 does.

A column-by-column profile is worth writing once and reusing for the rest of your career.



In [ ]:
profile <- function(df) {
  data.frame(
    column   = names(df),
    class    = vapply(df, function(s) class(s)[1], character(1)),
    non_null = vapply(df, function(s) sum(!is.na(s)), integer(1)),
    missing  = vapply(df, function(s) sum(is.na(s)), integer(1)),
    n_unique = vapply(df, function(s) length(unique(na.omit(s))), integer(1)),
    example  = vapply(df, function(s) {
      e <- na.omit(s); if (length(e)) substr(as.character(e[1]), 1, 34) else ""
    }, character(1)),
    row.names = NULL)
}
knitr::kable(profile(raw), caption = "Column profile of the raw ROCK1 table")



<p class="caption"><b>Table (Section 2).</b> Column profile of the raw table: class, how many of the 5,089 rows have a value, how many distinct values, and an example. <code>activity_id</code> has 5,089 distinct values — the primary key. <code>molecule_chembl_id</code> has 4,312, so the table is <em>not</em> one row per compound. <code>assay_chembl_id</code> has 276 and <code>document_chembl_id</code> 225: a few hundred experiments, each contributing many rows. <code>pref_name</code> is present in 538 rows — most compounds in a medicinal-chemistry paper never get a name — and <code>pchembl_value</code> is missing in 1,759, the most instructive missingness in the table (Section 7).</p>

Missingness deserves a picture rather than a column of numbers, because what matters is not how much is missing but *which rows* are missing together.



In [ ]:
miss <- is.na(raw)
frac <- sort(colMeans(miss), decreasing = TRUE)
frac <- frac[frac > 0]
d_bar <- tibble::tibble(column = factor(names(frac), levels = rev(names(frac))),
                        pct = 100 * as.numeric(frac))
p_bar <- ggplot(d_bar, aes(pct, column)) +
  geom_col(fill = PAL["orange"], width = 0.75) +
  geom_text(aes(label = sprintf("%.0f%%", pct)), hjust = -0.15, size = 2.5) +
  scale_x_continuous(limits = c(0, 118), expand = c(0, 0)) +
  labs(x = "% of the 5,089 rows missing", y = NULL, title = "Missing values per column")
d_map <- as.data.frame(miss[, names(frac), drop = FALSE]) |>
  mutate(row = row_number()) |>
  pivot_longer(-row, names_to = "column", values_to = "is_missing") |>
  mutate(column = factor(column, levels = names(frac)))
p_map <- ggplot(d_map, aes(column, row, fill = is_missing)) +
  geom_raster() +
  scale_fill_manual(values = c(`TRUE` = "grey15", `FALSE` = "grey92"), guide = "none") +
  scale_y_reverse(expand = c(0, 0)) +
  labs(x = NULL, y = "row (file order)", title = "Missingness map (dark = missing)") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6))
print(p_bar | p_map)
cat("columns with no missing value at all:", sum(colMeans(is.na(raw)) == 0), "of", ncol(raw), "\n")



<p class="caption"><b>Figure 2.</b> Missingness in the raw table. Left: the thirteen columns with gaps, as a percentage of 5,089 rows — <code>data_validity_comment</code> is empty in 99.7% of rows, which the label rounds to 100% (it is an exception flag, so emptiness is normal), <code>max_phase</code> 91%, <code>pref_name</code> 89%, <code>molecule_type</code> 38%, <code>pchembl_value</code> 35%, <code>units</code> 23%. Right: the map, one line per row, dark where missing. <code>units</code> shows solid dark bands — contiguous blocks of rows, i.e. a few source documents, with no unit at all. Contiguity in file order signals a <em>structural</em> cause; Section 9 shows those rows reported dimensionless pKi values.</p>

## 3. Grain: the question decides the row

Before any cleaning, decide the **grain** — what one row stands for. The raw file's grain is *one measurement*; the modelling table's is *one compound*. The conversion between them is irreversible: a compound measured five times becomes one row and the five are reduced to summary statistics. Confusing the two is the commonest error in bioactivity modelling, and its symptom is a model that scores beautifully because the same compound sits in both the training and the test set under two activity records.



In [ ]:
per_mol <- raw |> count(molecule_chembl_id, name = "n_meas")
cat("distinct compounds:", nrow(per_mol), " measurements:", sum(per_mol$n_meas), "\n")
print(summary(per_mol$n_meas))
cat("compounds measured once:", sum(per_mol$n_meas == 1),
    "| 2-5 times:", sum(per_mol$n_meas >= 2 & per_mol$n_meas <= 5),
    "| more than 5:", sum(per_mol$n_meas > 5), "\n")
cat("most-measured compound:", per_mol$molecule_chembl_id[which.max(per_mol$n_meas)],
    "with", max(per_mol$n_meas), "measurements\n")

counts <- per_mol |> count(n_meas, name = "n_compounds")
print(ggplot(counts, aes(n_meas, n_compounds)) +
  geom_col(fill = PAL["blue"], width = 0.75) +
  geom_text(data = subset(counts, n_meas %in% c(1, 2, 5, 10, 20)),
            aes(label = n_compounds), vjust = -0.4, size = 2.6) +
  scale_y_log10() +
  labs(x = "measurements for one compound", y = "number of compounds (log scale)",
       title = "The grain problem: most compounds appear once, a few appear often"))



<p class="caption"><b>Figure 3.</b> Measurements per compound (logarithmic vertical axis). 3,802 of the 4,312 compounds were measured exactly once, 491 between two and five times and 19 more than five times; the tail reaches 23 for the most-studied molecule, <code>CHEMBL4594250</code> (netarsudil) — typical of literature data, where interesting compounds get re-measured and the rest appear once. Any aggregation must behave sensibly at <em>n</em> = 1 and at <em>n</em> = 23.</p>

## 4. Reshaping the real table

`pivot_longer` has three partners you will reach for constantly.

- **`pivot_wider`** runs `pivot_longer` backwards, spreading one column into many. Unlike the old `spread`, it does not silently aggregate: if several rows land in one cell it returns list-columns and warns, which is a useful *assertion* when you believe the index/column pair is unique. When you do want aggregation, count or summarise first and then widen — the two steps are kept separate on purpose.
- **`nest`/`unnest`** move whole sub-tables into and out of a list-column, the tidyverse's answer to a MultiIndex: `nest` packs each group into one cell, `unnest` unpacks it again.
- **`separate_rows`** turns a column of delimited strings into one row per element — the standard way to normalise a packed collection into tidy form, and the verb we will need in Section 8 to read a `|`-separated list of identifiers.

The chunk below applies all of them to the real table.



In [ ]:
by_assay_type <- raw |>
  count(assay_chembl_id, standard_type) |>
  pivot_wider(names_from = standard_type, values_from = n, values_fill = 0)
cat("assays x measurement types:", nrow(by_assay_type), "x", ncol(by_assay_type) - 1, "\n")
cat("\nthe five largest assays:\n")
print(as.data.frame(by_assay_type |>
  mutate(total = IC50 + Ki + Kd) |> slice_max(total, n = 5)))

long_again <- by_assay_type |>
  pivot_longer(-assay_chembl_id, names_to = "standard_type", values_to = "n") |>
  filter(n > 0)
cat("\npivoted back to long:", nrow(long_again), "rows (the non-zero cells)\n")
cat("\ncounts cross-tabulated on two keys, then widened:\n")
print(as.data.frame(raw |> count(assay_type, standard_type) |>
  pivot_wider(names_from = standard_type, values_from = n, values_fill = 0)))



Three things to read off this output. The assay × type matrix is 276 × 3 and almost empty — each assay measures one quantity, not three, so of its 828 cells only 286 are non-zero, exactly the number of rows the round trip back to long form recovers. The largest single assay, `CHEMBL1963814`, contributes 1,019 Ki measurements alone; one experiment supplying a fifth of a dataset is a structural property no modelling can undo. And the `assay_type` × `standard_type` matrix shows binding assays (`B`) dominating with 3,252 IC50, 461 Ki and 347 Kd rows, while the functional assays (`F`) are almost entirely that one large Ki experiment and the ADMET rows (`A`) a negligible 9. A functional Ki and a binding Ki are not the same physical quantity, and here the distinction is confounded with "which assay".

`nest`/`unnest` is best shown as a round trip: pack the groups, then unpack them.



In [ ]:
packed <- raw |>
  select(molecule_chembl_id, assay_chembl_id) |>
  distinct() |>
  group_by(molecule_chembl_id) |>
  summarise(assays = list(sort(unique(assay_chembl_id))), n_assays = n(), .groups = "drop")
cat("packed: one row per compound, the assays in a list column\n")
print(head(as.data.frame(packed |> mutate(assays = sapply(assays, paste, collapse = ","))), 3))

unpacked <- packed |> select(-n_assays) |> unnest(assays)
cat("\nunnested:", nrow(unpacked), "rows = sum of the list lengths\n")
n_pairs <- nrow(distinct(raw, molecule_chembl_id, assay_chembl_id))
cat("distinct (compound, assay) pairs in the raw table:", n_pairs, "\n")
stopifnot(nrow(unpacked) == n_pairs)
cat("round trip verified by stopifnot()\n")
cat("\ncompounds tested in more than one assay:", sum(packed$n_assays > 1), "\n")



The `stopifnot()` is the point of the chunk, not the printout. Every reshaping operation has an invariant — here, "unnested rows = distinct compound–assay pairs", 4,862 of them — and writing it down converts a silent corruption into a loud failure. Make it a habit: one `stopifnot()` under every reshape, join and aggregation.<sup class="c"><a href="#r7">[7]</a></sup> Only 394 of the 4,312 compounds were ever tested in more than one assay, which is the first hint that this dataset's replicate structure is thin.

<div class="box bridge"><h4>Domain context — what an assay identifier buys you</h4>
<p>In ChEMBL an <strong>assay</strong> is one protocol from one paper: a specific enzyme construct, substrate, ATP concentration and detection method. Two assays for the same protein routinely give IC50 values differing fivefold, because IC50 depends on the ATP concentration (competitive inhibitors look weaker when ATP is abundant), the enzyme preparation and the readout chemistry. The assay identifier is therefore provenance, and "which assay" a legitimate grouping variable. The <code>assay_type</code> letter distinguishes <strong>B</strong> (binding: does the molecule stick?) from <strong>F</strong> (functional: does it stop the protein working?) and <strong>A</strong> (ADMET: absorption, distribution, metabolism, excretion, toxicity).</p></div>

## 5. Joining tables safely

Rule 3 says the assays belong in their own table. Let us build it, join it back, and then break the join on purpose.

A join is the relational operation Codd formalised in 1970,<sup class="c"><a href="#r28">[28]</a></sup> and three things go wrong with it: (i) the key does not mean the same thing on both sides; (ii) the *cardinality* is not what you assumed, so rows multiply; (iii) some keys do not match and you never notice. Modern dplyr has an argument for each — `relationship=` declares the cardinality and raises an error if it is violated, `unmatched=` decides whether an unmatched key is tolerated, and the choice of verb (`left_join`, `inner_join`, `anti_join`) decides the fate of non-matching rows. Use them, always. Readers who know pandas will recognise `relationship=` as the equivalent of `validate=`; dplyr goes one step further and *warns about an undeclared many-to-many relationship by default*, which pandas does not.



In [ ]:
assays <- raw |>
  group_by(assay_chembl_id) |>
  summarise(assay_type = first(assay_type),
            bao_label = first(bao_label),
            description = first(assay_description),
            document = first(document_chembl_id),
            n_measurements = n(),
            n_compounds = n_distinct(molecule_chembl_id),
            .groups = "drop")
cat("assay table:", nrow(assays), "rows x", ncol(assays), "columns\n")
cat("unique on the key:", !any(duplicated(assays$assay_chembl_id)), "\n")

joined <- raw |>
  left_join(select(assays, assay_chembl_id, assay_type_a = assay_type, n_compounds),
            by = "assay_chembl_id", relationship = "many-to-one")
cat("rows before:", nrow(raw), "| after:", nrow(joined),
    "| unmatched:", sum(is.na(joined$n_compounds)), "\n")
cat("the joined column agrees with the original:",
    all(joined$assay_type == joined$assay_type_a), "\n")



Row count unchanged, every row matched: that is what a correct many-to-one join looks like, and `relationship = "many-to-one"` is the line that makes the claim checkable. Note also the renamed incoming column (`assay_type_a =`): an explicit name beats dplyr's `.x`/`.y` suffixes three chunks later.

Now the failure mode. Suppose a colleague hands you an assay table that was built without de-duplication — a very common accident, because it is what you get if you select two columns from the measurement table and forget `distinct()`. The join silently produces far more rows than you started with.



In [ ]:
bad_lookup <- raw |> select(assay_chembl_id, assay_type_dup = assay_type)   # 5,089 rows, 276 keys
exploded <- suppressWarnings(
  raw |> left_join(bad_lookup, by = "assay_chembl_id", relationship = "many-to-many"))
cat("rows before:", nrow(raw), " -> after the careless join:", nrow(exploded),
    sprintf("(%.0fx)", nrow(exploded) / nrow(raw)), "\n")
cat("expected row count = sum of squared group sizes =",
    sum(count(raw, assay_chembl_id)$n^2), "\n")

guarded <- tryCatch(
  raw |> left_join(bad_lookup, by = "assay_chembl_id", relationship = "many-to-one"),
  error = function(e) conditionMessage(e))
cat("\nrelationship = 'many-to-one' stops it:\n", guarded, "\n")



A table of 5,089 rows became 1,647,711 — a 324-fold explosion — because the join paired every measurement with every row of the lookup that shared its assay. The resulting row count is exactly the sum of the squared group sizes, a quantity dominated by the largest group (1,019 rows contribute 1,038,361 on their own). Two safety nets fired: `relationship = "many-to-one"` turned the silent corruption into an error, and had we written no `relationship` at all, dplyr would still have warned us about an *unexpected many-to-many relationship* — which is why the demonstration above had to ask for `"many-to-many"` explicitly and wrap the call in `suppressWarnings()` to produce the broken table at all. Treat that warning as an error in your own code.

The third tool, the **anti-join**, answers "what is in A but not in B?". Here R is more direct than pandas: `anti_join()` is a verb of the language, not an idiom built from a merge indicator.



In [ ]:
exact_rows <- raw |> filter(coalesce(standard_relation, "=") == "=")
assays_with_exact <- exact_rows |> distinct(assay_chembl_id)
anti <- assays |> anti_join(assays_with_exact, by = "assay_chembl_id")
cat("assays in total:", nrow(assays),
    "| assays with at least one exact value:", nrow(assays_with_exact), "\n")
cat("assays that are entirely censored (anti-join result):", nrow(anti), "\n")
cat("they account for", sum(anti$n_measurements), "measurements\n")
print(as.data.frame(anti |> slice_max(n_measurements, n = 3) |>
  select(assay_chembl_id, assay_type, n_measurements, n_compounds)))



Sixty-one of the 276 assays never report a single exact number — every measurement is a "greater than" or a "less than". Those 61 experiments are small (93 measurements between them, the largest contributing 8), but the anti-join has found what no summary statistic would show: a whole class of experiment that disappears without trace the moment you filter the dataset to exact values, taking with it the information that a set of compounds was tested and found inactive. Across the full table the loss is far larger — 1,727 of the 5,089 measurements are censored. Section 10 returns to this.

Note the defensive detail in the filter: `coalesce(standard_relation, "=")`. Twenty-six rows have no relation symbol, and `standard_relation == "="` returns `NA` for them, which `filter()` treats as "drop" — the single most frequent source of quietly lost rows in dplyr pipelines.

<div class="exercise"><h4>Exercise 2</h4>
<p>Build the document-level companion table (one row per <code>document_chembl_id</code>, with its numbers of assays, compounds and measurements), join it back onto <code>raw</code> with <code>relationship=</code>, then use an <code>anti_join</code> to find the documents whose compounds appear in no other document. How many are there, and what fraction of the compounds do they hold? Hint: build the compound-to-document table first, then anti-join it against itself on a different key.</p></div>

## 6. Split–apply–combine: `group_by`, `summarise`, `mutate`, windows

Split–apply–combine<sup class="c"><a href="#r8">[8]</a></sup> names the pattern behind nearly every aggregation: split the table into groups, apply a function per group, combine the results. The tidyverse exposes three combination semantics, and choosing the wrong one is a frequent bug.

- **`group_by` + `summarise`** returns *one row per group* — a summary table. (This is pandas' `agg`.)
- **`group_by` + `mutate`** returns *one row per original row*, broadcasting the group result back — for normalising within a group or adding a group statistic as a column. (This is pandas' `transform`, and in R it is not a separate verb but the same `mutate` you already know, which is why R users rarely make the mistake of writing it as a join.)
- **`group_by` + `filter`** keeps *the original rows of the groups satisfying a predicate* — for dropping small groups.

Before aggregating we need a comparable number. Potency spans six orders of magnitude, so the field works on a logarithmic scale, **pActivity** \(= -\log_{10}\) (activity in mol/L), which for a value in nanomolar is

\[ \mathrm{pActivity} \;=\; -\log_{10}\!\left(\text{value}_{\mathrm{nM}} \times 10^{-9}\right) \;=\; 9 - \log_{10}\left(\text{value}_{\mathrm{nM}}\right). \]

IC50 = 1 nM gives pActivity 9, 1 µM gives 6, 10 µM gives 5: bigger is better and one unit is a tenfold change. ChEMBL's `standard_value` is already in nM, so we can compute the scale now; Section 9 rebuilds it from the *reported* column without ChEMBL's help and compares.



In [ ]:
work <- raw |>
  filter(!is.na(standard_value), standard_value > 0) |>
  mutate(pActivity = 9 - log10(standard_value))
cat("rows with a usable standard_value:", nrow(work), "of", nrow(raw), "\n")
cat(sprintf("pActivity range: %.2f to %.2f\n", min(work$pActivity), max(work$pActivity)))

summary_tbl <- work |>
  group_by(assay_chembl_id) |>
  summarise(n = n(),
            n_compounds = n_distinct(molecule_chembl_id),
            median_pAct = median(pActivity),
            iqr_pAct = IQR(pActivity),
            frac_censored = mean(coalesce(standard_relation, "=") != "="),
            assay_type = first(assay_type),
            .groups = "drop") |>
  arrange(desc(n))
cat("\nthe eight largest assays:\n")
print(as.data.frame(summary_tbl |> slice_head(n = 8) |>
  mutate(across(where(is.numeric), \(x) round(x, 3)))))



Read the table. The top assay, `CHEMBL1963814`, contributes 1,019 measurements over 1,002 distinct compounds with a median pActivity of 5.40 and **51% of its values censored** — half the compounds in it were simply inactive. The second, `CHEMBL5733564`, packs 413 measurements into an interquartile range of only 0.74 log units, and the sixth, `CHEMBL3991870`, is 93% censored with an interquartile range of exactly zero — 243 compounds all reported as "> 30 µM". These are not interchangeable experiments, and a model trained on their pooled values inherits the mixture.

Now the same split, combined the other way.



In [ ]:
work <- work |>
  group_by(assay_chembl_id) |>
  mutate(assay_median = median(pActivity),
         assay_n = n(),
         rank_in_assay = min_rank(desc(pActivity))) |>
  ungroup() |>
  group_by(molecule_chembl_id) |>
  mutate(nth_measurement = row_number()) |>
  ungroup() |>
  mutate(pAct_centred = pActivity - assay_median)

print(as.data.frame(work |> slice_max(assay_n, n = 6) |>
  select(molecule_chembl_id, assay_chembl_id, pActivity, assay_median,
         pAct_centred, rank_in_assay, nth_measurement) |>
  mutate(across(where(is.numeric), \(x) round(x, 2)))))
cat(sprintf("\nspread of raw pActivity:      sd = %.3f\n", sd(work$pActivity)))
cat(sprintf("spread after assay-centring: sd = %.3f\n", sd(work$pAct_centred)))
cat(sprintf("variance explained by 'which assay': %.1f%%\n",
            100 * (1 - var(work$pAct_centred) / var(work$pActivity))))



Fifty-three point nine per cent of the variance of the potency values is explained by *which assay the measurement came from*. That number is the quantitative form of the warning in the domain box: the assay is not neutral metadata.

A caution on window functions: `min_rank`, `row_number`, `cumsum`, `lag` and `lead` all depend on **row order**, which is whatever the last operation left behind. `nth_measurement` counts a compound's measurements *in file order*, meaningful only because the file is in `activity_id` order; `arrange()` explicitly before any window operation. And always `ungroup()` when the grouped phase ends: a frame still grouped three pipes later is R's characteristic silent bug, because every later `mutate` and `filter` quietly operates per group.



In [ ]:
top12 <- summary_tbl |> slice_head(n = 12) |> pull(assay_chembl_id)
sub <- work |> filter(assay_chembl_id %in% top12)
ord <- sub |> group_by(assay_chembl_id) |> summarise(m = median(pActivity)) |> arrange(m) |>
  pull(assay_chembl_id)
sub <- sub |> mutate(assay = factor(str_remove(assay_chembl_id, "CHEMBL"),
                                    levels = str_remove(ord, "CHEMBL")))
p1 <- ggplot(sub, aes(assay, pActivity)) +
  geom_boxplot(fill = PAL["blue"], alpha = 0.75, outlier.shape = NA, linewidth = 0.3) +
  labs(x = "assay (ChEMBL id, prefix dropped)", y = "pActivity", title = "raw pActivity by assay") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6.5))
p2 <- ggplot(sub, aes(assay, pAct_centred)) +
  geom_boxplot(fill = PAL["green"], alpha = 0.75, outlier.shape = NA, linewidth = 0.3) +
  geom_hline(yintercept = 0, linetype = "dashed", colour = "grey30", linewidth = 0.35) +
  labs(x = "assay (ChEMBL id, prefix dropped)", y = "pActivity \u2212 assay median",
       title = "after subtracting the assay median") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6.5))
print(p1 | p2)



<p class="caption"><b>Figure 4.</b> The twelve largest assays, before and after centring. Left: the assay medians span roughly two and a half log units, from 4.5 to 7.4, so apparent potency depends materially on which experiment measured the compound. Right: after centring, the boxes align on zero by construction and what remains is the within-assay spread, from essentially zero (the all-censored panel <code>CHEMBL3991870</code>) to more than a log unit either side.</p>

## 7. Missing data: three mechanisms, three responses

`NA` is not a value but the absence of one, and the right response depends on *why* it is absent. Rubin's three cases<sup class="c"><a href="#r9">[9]</a></sup> are worth memorising.

- **MCAR** (missing completely at random): the probability of being missing depends on nothing — a tube was dropped. Dropping incomplete rows loses precision but introduces no bias.
- **MAR** (missing at random, a badly chosen name): the probability depends on *observed* variables but not on the missing value itself. Dropping rows biases the result; conditioning on those variables — stratification, regression, multiple imputation — recovers it.<sup class="c"><a href="#r10">[10]</a></sup>
- **MNAR** (missing not at random): the probability depends on the unobserved value — a measurement is absent *because it was too small to measure*. No imputation fixes this from the data alone; the mechanism must be modelled or the missingness carried through as information.

The three cannot be told apart by looking at the missing cells — they are, after all, missing — only by understanding the data-generating process. Our table contains a textbook MNAR column.



In [ ]:
ct <- table(relation = coalesce(raw$standard_relation, "(missing)"),
            pchembl = ifelse(is.na(raw$pchembl_value), "pchembl missing", "pchembl present"))
print(ct)
cat("\ntotal pchembl_value missing:", sum(is.na(raw$pchembl_value)), "of", nrow(raw),
    sprintf("(%.0f%%)", 100 * mean(is.na(raw$pchembl_value))), "\n")
cat("missing among exact '=' rows:",
    sum(raw$standard_relation == "=" & is.na(raw$pchembl_value), na.rm = TRUE), "\n")



The pattern is absolute. Every one of the 1,410 rows whose relation is `>`, all 312 whose relation is `<`, all 5 `<=` rows and all 26 rows with no relation at all lack a `pchembl_value`; among the 3,336 exact rows only 6 lack one. ChEMBL does not compute its logarithmic potency for censored measurements, because "> 10 µM" has no single logarithm. So the missingness of `pchembl_value` is *caused by the value being unmeasurable*, which is MNAR in its purest form — and it is biased in a specific direction: the missing rows are the **weak** compounds. Fit a model on complete cases and you have fitted it on a dataset from which most inactives were deleted.

Note that `table()` drops `NA` by default, which is why the relation column was passed through `coalesce()` first: a contingency table that silently omits a quarter of the rows is worse than no table at all. The simulation below makes the cost visible by removing values from a column we fully observe, under the three mechanisms, and estimating the same mean each time.



In [ ]:
set.seed(0)
full <- work |> filter(standard_relation == "=") |> select(pActivity, assay_n) |> tidyr::drop_na()
y <- full$pActivity
big_assay <- full$assay_n > 200                      # an OBSERVED covariate
truth <- mean(y)

p_mcar <- rep(0.35, length(y))                       # independent of everything
p_mar  <- ifelse(big_assay, 0.60, 0.15)              # depends on an observed variable only
p_mnar <- 1 / (1 + exp(2.0 * (y - 6.0)))             # depends on the value itself

res <- lapply(list(MCAR = p_mcar, MAR = p_mar, MNAR = p_mnar), function(p) {
  ests <- replicate(200, mean(y[runif(length(y)) > p]))
  data.frame(mean_fraction_deleted = mean(p), complete_case_mean = mean(ests),
             bias = mean(ests) - truth, sd_of_estimate = sd(ests))
})
res <- cbind(mechanism = names(res), do.call(rbind, res))
cat(sprintf("true mean pActivity of the full set: %.3f\n", truth))
print(res, row.names = FALSE, digits = 3)



Under MCAR the complete-case mean is unbiased: deleting a third of the rows at random costs precision but leaves the estimate centred on the truth. Under MAR, deleting preferentially from the large assays shifts it by 0.06 log units — but the driving covariate is observed, so a stratified or weighted estimate repairs it. Under MNAR the estimate is biased upward by roughly half a log unit and *nothing in the observed data reveals it*. That is the situation `pchembl_value` puts us in, and the reason we keep censored rows and flag them (Section 10).



In [ ]:
work <- work |> mutate(rel_std = coalesce(standard_relation, "(missing)"),
                       kind = ifelse(rel_std == "=", "exact (=)", "censored (>, <, <=)"))
p1 <- ggplot(work, aes(pActivity, fill = kind)) +
  geom_histogram(binwidth = 0.25, boundary = 2.5, colour = NA) +
  scale_fill_manual(values = c("exact (=)" = unname(PAL["blue"]),
                               "censored (>, <, <=)" = unname(PAL["orange"])), name = NULL) +
  labs(x = "pActivity", y = "measurements",
       title = "Censored rows are concentrated at low potency") +
  theme(legend.position = "bottom", legend.key.size = unit(0.3, "cm"))
d2 <- raw |>
  mutate(rel = coalesce(standard_relation, "(missing)"),
         pch = ifelse(is.na(pchembl_value), "pchembl missing", "pchembl present")) |>
  count(rel, pch)
p2 <- ggplot(d2, aes(n, rel, fill = pch)) +
  geom_col(width = 0.7) +
  scale_fill_manual(values = c("pchembl missing" = unname(PAL["orange"]),
                               "pchembl present" = unname(PAL["blue"])), name = NULL) +
  labs(x = "rows", y = "standard_relation",
       title = "pchembl_value is missing exactly when censored") +
  theme(legend.position = "bottom", legend.key.size = unit(0.3, "cm"))
print(p1 | p2)



<p class="caption"><b>Figure 5.</b> The missingness of <code>pchembl_value</code> is not random. Left: pActivity for exact (blue) and censored (orange) measurements — the censored mass sits at low potency with a tall spike at pActivity 5, "&gt; 10 µM", the standard top concentration of a screen. Right: the contingency bar chart — every <code>&gt;</code>, <code>&lt;</code> and <code>&lt;=</code> row lacks a <code>pchembl_value</code>, every <code>=</code> row but six has one. Restricting an analysis to rows with a <code>pchembl_value</code> restricts it to the potent compounds.</p>

## 8. Duplicates: by key, by flag, by structure

"Remove duplicates" is three operations that disagree with each other. Answer first: *duplicate with respect to what?*

1. **Exact-key duplicates** — rows agreeing on the columns that identify a measurement (same compound, quantity, number, assay), almost always an extraction artefact.
2. **Flagged duplicates** — ChEMBL's `potential_duplicate` column marks rows it suspects of being re-reports of one experiment, typically across documents.
3. **Structural duplicates** — *different identifiers, same molecule*: a free base and its hydrochloride salt, or two deposits of one structure. Invisible to any identifier-based test; they require chemistry.



In [ ]:
key <- c("molecule_chembl_id", "standard_type", "standard_value", "assay_chembl_id")
dup_any <- duplicated(raw[key]) | duplicated(raw[key], fromLast = TRUE)
cat("rows belonging to an exact-key duplicate group:", sum(dup_any),
    "| redundant rows (all but the first of each group):", sum(duplicated(raw[key])), "\n")
cat("rows flagged potential_duplicate by ChEMBL:", sum(raw$potential_duplicate == 1, na.rm = TRUE), "\n")
cat("rows that are BOTH exact-key duplicates and flagged:",
    sum(dup_any & raw$potential_duplicate == 1, na.rm = TRUE), "\n")
cat("\none duplicate group, side by side:\n")
print(as.data.frame(raw[dup_any, ] |> arrange(across(all_of(key))) |>
  select(all_of(key), activity_id, document_chembl_id, potential_duplicate) |> head(2)))



The two tests barely overlap: 146 rows form exact-key duplicate groups (78 redundant), 279 rows carry ChEMBL's flag, and only 4 rows are in both sets. They detect different things — the key test finds identical records within one assay, the flag marks values believed to be copied between publications. Neither is wrong; using one and calling the data de-duplicated is. Note the base-R idiom `duplicated(x) | duplicated(x, fromLast = TRUE)`: `duplicated()` alone marks the second and later copies, so to see *all* members of each group you must scan from both ends. `dplyr::distinct()` keeps the first of each group, equivalent to dropping `duplicated(x)`.

The third kind needs chemistry: a molecule may be deposited as a free base, a hydrochloride salt or a hydrate, each with its own ChEMBL identifier and all the same compound for modelling. The solution is a canonical identifier of the *parent* structure.

<div class="box bridge"><h4>Domain context — SMILES, standardisation and the InChIKey</h4>
<p>A <strong>SMILES</strong> string is a molecule written as text: atoms are letters, bonds are implicit or written as symbols, rings are opened and closed by matching digits.<sup class="c"><a href="#r11">[11]</a></sup> The same molecule can be written as many different valid SMILES, so software <em>canonicalises</em> them. A <strong>salt</strong> carries a counter-ion so the drug dissolves better; the dot in <code>...CC1.Cl</code> separates the two fragments. <strong>Standardisation</strong> strips the counter-ion, neutralises charges and normalises functional-group drawings.<sup class="c"><a href="#r12">[12]</a></sup> The <strong>InChIKey</strong> is a 27-character hash of a standard structure description<sup class="c"><a href="#r13">[13]</a></sup> — two molecules with the same InChIKey are the same molecule, which makes it the natural key for de-duplicating chemical tables.</p></div>

<div class="box"><h4>Substitution in the R edition — where the InChIKeys come from</h4>
<p>The Python edition standardises the 4,309 distinct SMILES strings with RDKit (<code>Normalizer</code>, <code>LargestFragmentChooser</code>, <code>Uncharger</code>) and computes the InChIKey in the notebook. R has no RDKit binding in this environment, so this unit <strong>reads the result of that standardisation</strong> from the frozen reference table <code>rock1_chembl_curated.csv</code>, whose <code>chembl_ids</code> column lists every ChEMBL identifier that collapsed into each standardised parent. Recovering the identifier-to-InChIKey map is a <code>separate_rows()</code> exercise, which is why this section is also where that verb earns its place. The structures themselves ship as <code>data/rock1_curated.sdf.gz</code> and the generation recipe is recorded in <code>data/manifest_fingerprints.json</code>, so a reader with ChemmineR or rcdk can redo the chemistry; nothing here depends on it.</p></div>



In [ ]:
curated <- load_data("rock1_chembl_curated.csv")
key_map <- curated |>
  select(inchikey, std_smiles, chembl_ids) |>
  separate_rows(chembl_ids, sep = "\\|") |>
  rename(molecule_chembl_id = chembl_ids)
cat("map rows:", nrow(key_map),
    "| distinct ChEMBL ids:", n_distinct(key_map$molecule_chembl_id),
    "| distinct InChIKeys:", n_distinct(key_map$inchikey), "\n")
cat("raw ChEMBL ids absent from the map (dropped by the protocol):",
    sum(!unique(raw$molecule_chembl_id) %in% key_map$molecule_chembl_id), "\n")
collapsed <- key_map |> count(inchikey, name = "n_ids")
cat("InChIKeys covering more than one ChEMBL id:", sum(collapsed$n_ids > 1),
    "| identifiers they absorb:", sum(collapsed$n_ids[collapsed$n_ids > 1]), "\n")



Nine InChIKeys cover nineteen ChEMBL identifiers, and 22 of the raw file's 4,312 identifiers never reach the map at all — those are the rows the reference protocol removed (no structure, a validity flag, no numeric value, or a molecule outside the 3–70 heavy-atom window). A small number of collapses — but look at *which* compounds.



In [ ]:
multi <- collapsed$inchikey[collapsed$n_ids > 1]
view <- raw |>
  inner_join(filter(key_map, inchikey %in% multi), by = "molecule_chembl_id",
             relationship = "many-to-one") |>
  distinct(inchikey, molecule_chembl_id, pref_name) |>
  arrange(inchikey, molecule_chembl_id) |>
  mutate(inchikey = str_sub(inchikey, 1, 14))
print(as.data.frame(view))



Ripasudil, a marketed ROCK inhibitor used as eye drops for glaucoma, appears three times — free base, hydrochloride and hydrochloride dihydrate — each with its own identifier and its own measurements; tofacitinib appears as free base and citrate, netarsudil as free base and dihydrochloride, and Y-27632 shares its parent with an unnamed deposit. De-duplicate on `molecule_chembl_id` and these stay separate, so one molecule can land in both training and test set — a leak that inflates every validation metric you report. De-duplicate on the InChIKey of the standardised parent and they merge. This is why structure standardisation is step one of every published curation protocol,<sup class="c"><a href="#r14">[14]</a></sup><sup class="c"><a href="#r15">[15]</a></sup> why audits repeatedly find wrong or duplicated structures in modelling datasets,<sup class="c"><a href="#r34">[34]</a></sup> and why cross-database identifier mapping is itself a curated service.<sup class="c"><a href="#r33">[33]</a></sup>

## 9. Units: an explicit conversion table beats a clever regex

The `units` column is the archetype of a field that looks categorical and is not. Let us look at it before deciding anything.



In [ ]:
cat("reported units:\n");            print(table(raw$units, useNA = "ifany"))
cat("\nreported measurement types:\n"); print(table(raw$type, useNA = "ifany"))
cat("\nChEMBL's standardised vocabulary:\n")
print(table(raw$standard_units, useNA = "ifany"))
print(table(raw$standard_type, useNA = "ifany"))



Seven unit strings, four of which (`10'-8M`, `10'-10M`, `10^-10M`, `10^-9M`) are not units but scientific notation leaked from a paper's table header; 1,167 rows have no unit; `type` has seven values where `standard_type` has three. Two of the seven — `pKi` (1,026 rows) and `pIC50` (115 rows) — are *already logarithms*, which is why their rows carry no unit: a logarithm is dimensionless. `Kd apparent` (243) and `Kd(app)` (2) are two spellings of one thing.

Write the mapping as **data, not code**: a named vector from unit string to a multiplicative factor into a base unit, plus an explicit vector of already-logarithmic types. Anything absent from the table stays `NA` and is counted, never silently coerced. The single regular expression we allow ourselves, for the scientific-notation strings, is written to fail loudly.

<h4>Worked example by hand</h4>

Take row 2,671 of the file, which reports `type = IC50`, `value = 7.99`, `units = 10'-10M`. The header means "the number in this column is in units of 10<sup>−10</sup> mol/L". Therefore

| step | expression | result |
|---|---|---|
| 1. read the exponent out of the unit string | `10'-10M` → exponent −10 | −10 |
| 2. convert the unit to nanomolar | 10<sup>−10</sup> mol/L = 10<sup>−10</sup> × 10<sup>9</sup> nM = 10<sup>−1</sup> nM | factor 0.1 |
| 3. apply the factor | 7.99 × 0.1 | 0.799 nM |
| 4. convert to molar | 0.799 × 10<sup>−9</sup> | 7.99 × 10<sup>−10</sup> M |
| 5. take the negative logarithm | −log<sub>10</sub>(7.99 × 10<sup>−10</sup>) = 9 − log<sub>10</sub>(0.799) | 9.097 |

Our conversion table should therefore turn this row into 0.799 nM, and ChEMBL's own `standard_value` for the row should agree. The next chunk builds the table and checks every convertible row at once. (Be careful with the index: R counts from 1, so the row Python calls 2670 is `raw[2671, ]` here — an off-by-one that has cost this laboratory more than one afternoon.)



In [ ]:
TO_NM <- c("nM" = 1, "uM" = 1e3, "\u00b5M" = 1e3, "mM" = 1e6, "M" = 1e9,
           "pM" = 1e-3, "fM" = 1e-6, "nmol/L" = 1, "umol/L" = 1e3, "mol/L" = 1e9)
LOG_TYPES <- c("pKi", "pIC50", "pKd")     # value is already -log10(molar): dimensionless
SCI <- "^10[\\^'](-?\\d+)\\s*M$"          # '10^-9M', "10'-8M"  ->  exponent

unit_factor <- function(u) {
  u <- trimws(u)
  f <- unname(TO_NM[u])                                   # NA for anything not in the table
  sci <- str_match(u, SCI)[, 2]
  hit <- !is.na(sci)
  f[hit] <- 10^(as.numeric(sci[hit]) + 9)
  f
}
raw <- raw |>
  mutate(unit_fac = unit_factor(units),
         is_log = type %in% LOG_TYPES,
         mine_nM = ifelse(is_log, 10^(9 - value), unit_fac * value))
cat("distinct unit strings seen:", n_distinct(na.omit(raw$units)),
    "| converted by the table:", sum(!is.na(raw$unit_fac)),
    "| unknown unit strings:", sum(!is.na(raw$units) & is.na(raw$unit_fac)), "\n")
cat("rows handled as already-logarithmic types:", sum(raw$is_log), "\n")
cat("rows we could not convert at all:", sum(is.na(raw$mine_nM)), "->\n")
print(table(raw$activity_comment[is.na(raw$mine_nM)], useNA = "ifany"))
r <- raw[2671, ]
cat(sprintf("\nworked example, row 2671: value=%g units=%s -> ours %.3f nM | ChEMBL %.3f nM | pActivity %.3f\n",
            r$value, r$units, r$mine_nM, r$standard_value, 9 - log10(r$mine_nM)))



Hand calculation and code agree: 0.799 nM, pActivity 9.097, and ChEMBL's standardised value is also 0.799 nM. The 26 unconvertible rows are precisely those whose `value` is empty and whose `activity_comment` reads "Not Determined" — a missing-data problem, not a units problem, dropped in Unit 6 at the "no numeric value" step. Note that `unit_factor()` is vectorised through name-based lookup (`TO_NM[u]`), not through a loop: a named vector *is* R's dictionary, and indexing it by a character vector returns `NA` for every unknown key, which is exactly the loud-failure behaviour we wanted.

Now the real test: we reconstructed the measurement in nanomolar from the *reported* columns using only our own table, and ChEMBL did the same job independently. Do we agree?



In [ ]:
cmp <- raw |>
  filter(!is.na(mine_nM), !is.na(standard_value), standard_value > 0) |>
  mutate(log_diff = log10(mine_nM) - log10(standard_value),
         unit_label = coalesce(units, "(none: pKi/pIC50)"))
cat("comparable rows:", nrow(cmp), "\n")
cat("agree to within 0.001 log units:", sum(abs(cmp$log_diff) < 1e-3),
    "| disagree by more:", sum(abs(cmp$log_diff) >= 1e-3), "\n")
cat(sprintf("largest absolute disagreement: %.2e log units\n", max(abs(cmp$log_diff))))

lab <- cmp |> count(unit_label) |> mutate(lab = paste0(unit_label, " (n=", n, ")"))
cmp <- cmp |> left_join(select(lab, unit_label, lab), by = "unit_label", relationship = "many-to-one")
p1 <- ggplot(cmp, aes(standard_value, mine_nM, colour = lab)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", linewidth = 0.4, colour = "grey30") +
  geom_point(size = 0.7, alpha = 0.45) +
  scale_x_log10(limits = c(1e-3, 1e8)) + scale_y_log10(limits = c(1e-3, 1e8)) +
  scale_colour_manual(values = grDevices::colorRampPalette(unname(PAL))(nrow(lab)),
                      name = NULL) +
  labs(x = "ChEMBL standard_value (nM)", y = "our harmonised value (nM)",
       title = "Two independent harmonisations of the same numbers") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6),
        legend.key.size = unit(0.25, "cm"))
p2 <- ggplot(cmp, aes(log_diff)) +
  geom_histogram(bins = 41, fill = PAL["blue"]) + scale_y_log10() +
  labs(x = "log10(ours) \u2212 log10(ChEMBL)", y = "rows (log scale)",
       title = "Disagreement is rounding, not arithmetic")
print(p1 | p2)



<p class="caption"><b>Figure 6.</b> Left: our harmonised value against ChEMBL's, both in nanomolar, on logarithmic axes spanning eleven orders of magnitude; every point lies on the diagonal. The micromolar rows sit there only because the factor of 1,000 was applied — omit it and 470 points move three decades upward. Right: the log-difference for all 5,062 comparable rows, every one below 0.001 and the largest 5.71 × 10<sup>−5</sup>; the residue comes entirely from ChEMBL storing the back-converted pKi rows rounded to two decimals (a reported pKi of 8.7 becomes 1.995 nM where exact arithmetic gives 1.995262). Agreement with an independent curation does not prove either curation right, only that the arithmetic is sound and no unit was dropped — empirically where the errors are, since audits find error rates of a few per cent concentrated in unit and transcription mistakes.<sup class="c"><a href="#r16">[16]</a></sup><sup class="c"><a href="#r17">[17]</a></sup></p>

<div class="exercise"><h4>Exercise 3</h4>
<p>Delete <code>"uM" = 1e3</code> from <code>TO_NM</code> and re-run the two chunks above. How many rows become unconvertible, and what happens to the scatter plot if you instead treat the micromolar numbers as nanomolar (set the factor to 1)? The difference in their median pActivity is the size of the error a silent unit bug introduces.</p></div>

## 10. Censored measurements and the relation that flips

A censored measurement is one the experiment could not pin down: the compound was inactive at the highest concentration tested, so all we know is that its IC50 is *greater than* that concentration. The information is real but it is an inequality, not a number. Statistics has a mature theory for this — Tobin's censored regression,<sup class="c"><a href="#r18">[18]</a></sup> survival analysis, and an environmental-chemistry literature on the damage done by substituting values for non-detects<sup class="c"><a href="#r19">[19]</a></sup> — and in this unit our job is simply never to lose the flag.

One subtlety catches everyone once: on a logarithmic scale the inequality **reverses**, because \(-\log_{10}\) is a decreasing function. "pKi < 5" means "Ki > 10,000 nM". ChEMBL performs the flip when it standardises; the contingency table shows it happening.



In [ ]:
print(table(reported = coalesce(raw$relation, "(missing)"),
            standardised = coalesce(raw$standard_relation, "(missing)")))
flip <- raw |> filter(relation == "<", standard_relation == ">")
cat("\nrows whose relation flipped '<' -> '>':", nrow(flip), "\n")
print(table(flip$type))
print(as.data.frame(flip |> select(type, relation, value, units, standard_type,
                                  standard_relation, standard_value) |> head(3)))



Five hundred and fifty rows flip, all `pKi` (518) or `pIC50` (32): a paper writing "pKi < 5.0" meant "weaker than 10 µM", standardised as "Ki > 10,000 nM". Copy the `relation` column across unchanged and you invert 550 measurements, turning the weakest compounds into the strongest. Convert value and relation in the *same* function.

<h4>Worked example by hand (continued)</h4>

Row with `type = pKi`, `relation = "<"`, `value = 5.0`. Then Ki = 10<sup>−5.0</sup> M = 10,000 nM, and because \(-\log_{10}\) is decreasing, `<` on the p-scale becomes `>` on the concentration scale: **Ki > 10,000 nM**. On the pActivity scale we report back to the user, the value is 5.0 and the relation is `<` again — pActivity *is* the p-scale. The round trip is its own check.



In [ ]:
FLIP <- c("<" = ">", ">" = "<", "<=" = ">=", ">=" = "<=", "=" = "=")

to_nanomolar <- function(value, units, mtype, relation) {
  rel <- coalesce(relation, "=")
  islog <- mtype %in% LOG_TYPES
  list(nM  = ifelse(islog, 10^(9 - value), unit_factor(units) * value),
       rel = ifelse(islog, unname(FLIP[rel]), rel))
}
cases <- tibble::tibble(value = c(5, 7.99, 470, 10),
                        units = c(NA, "10'-10M", "uM", "uM"),
                        type  = c("pKi", "IC50", "Kd", "IC50"),
                        relation = c("<", "=", "=", ">"))
conv_cases <- to_nanomolar(cases$value, cases$units, cases$type, cases$relation)
print(as.data.frame(cases |> mutate(nM = round(conv_cases$nM, 3), rel_out = conv_cases$rel)))

conv <- to_nanomolar(raw$value, raw$units, raw$type, raw$relation)
raw <- raw |> mutate(value_nM = conv$nM, rel = conv$rel)
cat("\nour relation matches ChEMBL's standardised relation in",
    sum(raw$rel == coalesce(raw$standard_relation, "=")), "of", nrow(raw), "rows\n")



All 5,089 relations agree once the flip is part of the conversion function. The printed table also shows a Kd of 470 µM — so weak it is almost certainly not a real binding constant, and indeed ChEMBL flags that row "Outside typical range": harmonisation does not make a bad number good. Observe the shape of `to_nanomolar()`: it takes four vectors and returns a list of two vectors, so value and relation are computed by one call and cannot drift apart. A function that returned only the value would invite exactly the bug this section is about.

Finally, the picture that justifies keeping censored rows.



In [ ]:
raw <- raw |> mutate(pAct = ifelse(!is.na(value_nM) & value_nM > 0, 9 - log10(value_nM), NA_real_),
                     rel_std = coalesce(standard_relation, "(missing)"))
d1 <- raw |> filter(rel_std %in% c("=", ">", "<"), !is.na(pAct)) |>
  mutate(rel_lab = paste0("relation '", rel_std, "'"))
p1 <- ggplot(d1, aes(pAct, colour = rel_lab)) +
  geom_freqpoly(binwidth = 0.2, linewidth = 0.6) +
  scale_colour_manual(values = unname(PAL[c("blue", "green", "orange")]), name = NULL) +
  labs(x = "pActivity", y = "measurements",
       title = "Censored values pile up at the screening limits") +
  theme(legend.position = "bottom", legend.key.size = unit(0.3, "cm"))
pile <- raw |> filter(rel_std == ">") |> count(pAct = round(pAct, 3)) |>
  slice_max(n, n = 6) |> arrange(pAct)
p2 <- ggplot(pile, aes(factor(sprintf("%.2f", pAct)), n)) +
  geom_col(fill = PAL["orange"], width = 0.7) +
  geom_text(aes(label = n), vjust = -0.3, size = 2.6) +
  labs(x = "pActivity of '>' measurements (most frequent values)", y = "measurements",
       title = "The six commonest censoring limits")
print(p1 | p2)
cat("measurements censored:", sum(raw$rel_std %in% c(">", "<", "<=")), "of", nrow(raw),
    sprintf("(%.0f%%)\n", 100 * mean(raw$rel_std %in% c(">", "<", "<="))))
print(data.frame(pActivity = sprintf("%.3f", pile$pAct),
                 nM = sprintf("%.0f", 10^(9 - pile$pAct)), rows = pile$n))



<p class="caption"><b>Figure 7.</b> Left: pActivity by relation. Exact values (blue) form a broad distribution centred near 6.5; "greater than" values (orange) are narrow spikes, not a distribution — they are properties of the experiment, the highest concentration used. Right: the six commonest spikes, dominated by pActivity 5.00 ("&gt; 10 µM", 724 measurements) and 4.523 ("&gt; 30 µM", 340 measurements, the all-censored panel of Section 6). Treating the spikes as potencies, which is what dropping the relation column does, invents a cluster of identically active compounds that does not exist. In total 1,727 of 5,089 measurements (34%) are censored.</p>

## 11. Strings, regular expressions, factors and dates

Three mechanical skills remain.

**String operations.** `stringr` wraps the `stringi` engine in a consistent `str_*` family — `str_detect`, `str_match`, `str_extract`, `str_split`, `str_trim`, `str_to_lower` — all vectorised and all `NA`-safe, replacing the `for` loop you were about to write. The free-text `assay_description` is where they earn their keep: it is the only record of the experimental conditions that drive the between-assay variation measured in Section 6.



In [ ]:
desc <- coalesce(assays$description, "")
feat <- tibble::tibble(
  assay_chembl_id = assays$assay_chembl_id,
  n_measurements  = assays$n_measurements,
  len_chars       = str_length(desc),
  is_htrf         = str_detect(desc, regex("\\bHTRF\\b", ignore_case = TRUE)),
  is_radiometric  = str_detect(desc, regex("33P|32P|radiometric|scintillation", ignore_case = TRUE)),
  is_cell_based   = str_detect(desc, regex("cell|HUVEC|HeLa|smooth muscle", ignore_case = TRUE)),
  mentions_ATP    = str_detect(desc, "\\bATP\\b"),
  first_conc      = as.numeric(str_match(desc, "(\\d+\\.?\\d*)\\s*(?:\u03bc|\u00b5|u)M")[, 2]))
cat(sprintf("assay descriptions: median length %.0f characters, longest %d\n",
            median(feat$len_chars), max(feat$len_chars)))
print(colSums(feat[c("is_htrf", "is_radiometric", "is_cell_based", "mentions_ATP")]))
cat("\nassays quoting a micromolar concentration in the text:", sum(!is.na(feat$first_conc)),
    sprintf("| median value quoted: %.1f uM\n", median(feat$first_conc, na.rm = TRUE)))
cat("\nmeasurements contributed by each technology family:\n")
print(as.data.frame(feat |> group_by(is_htrf, is_radiometric) |>
  summarise(assays = n(), measurements = sum(n_measurements), .groups = "drop")))



Three regex habits. In R, backslashes must be doubled inside a string literal (`"\\bATP\\b"` is the pattern `\bATP\b`) unless you use the raw-string syntax `r"(\bATP\b)"` of R 4.0 — the most common difference from Python's `r"..."` prefix, and forgetting it produces a pattern that matches nothing. Anchor word boundaries with `\b`, or `ATP` matches inside `ATPase`. Prefer `str_match` with a capture group over `str_extract_all` when you expect one match: column 2 of the returned matrix is index-aligned, with `NA` where nothing matched. And µ appears in two Unicode forms (U+00B5 micro sign, U+03BC Greek mu), which is why the pattern lists both plus the ASCII fallback `u`.

Only 7 assays use HTRF and 28 a radiometric readout; 65 of the 276 descriptions mention ATP, the covariate that most strongly shifts a kinase IC50, and 33 quote a micromolar concentration in free text, median 10 µM. Free-text mining of this kind is how a reviewer's question — "were these IC50 values measured at the same ATP concentration?" — gets an answer at all.

**Factors.** A column with few distinct values and many rows is a candidate for a factor: R stores the levels once and an integer code per row. The saving is real but smaller than the corresponding saving in pandas, because R already interns character strings in a global cache, so repeated strings share storage before you do anything.

**Dates.** ChEMBL's activity endpoint returns none, so the chunk below *creates a synthetic deposition date* per document with a fixed seed — a teaching device, explicitly not a property of the real data — and demonstrates parsing and the `lubridate` accessors.<sup class="c"><a href="#r35">[35]</a></sup>



In [ ]:
cols4 <- c("assay_chembl_id", "standard_type", "document_chembl_id", "target_organism")
before <- sum(vapply(raw[cols4], function(x) as.numeric(object.size(x)), numeric(1)))
after  <- sum(vapply(lapply(raw[cols4], factor), function(x) as.numeric(object.size(x)), numeric(1)))
cat(sprintf("four identifier columns: %.2f MB as character -> %.2f MB as factor (%.1fx smaller)\n",
            before / 1e6, after / 1e6, before / after))
f2 <- raw |> mutate(across(all_of(c("assay_type", "standard_type")), factor))
cat("combinations actually observed      :", nrow(count(f2, assay_type, standard_type)), "\n")
cat("combinations with .drop = FALSE     :",
    nrow(count(f2, assay_type, standard_type, .drop = FALSE)), "\n")

set.seed(42)   # SYNTHETIC dates - the ChEMBL activity endpoint returns none. Teaching device only.
docs <- sort(unique(na.omit(raw$document_chembl_id)))
dep <- tibble::tibble(document_chembl_id = docs,
                      deposited = as.Date("2005-01-01") + sample(0:7299, length(docs), replace = TRUE))
dated <- raw |> left_join(dep, by = "document_chembl_id", relationship = "many-to-one")
cat("\nlubridate accessors:", min(year(dated$deposited)), "to", max(year(dated$deposited)),
    "| median year", median(year(dated$deposited)), "\n")
cat("measurements per year (synthetic dates), first five:\n")
print(as.data.frame(dated |> mutate(y = year(deposited)) |> count(y) |> head(5)))



The four identifier columns shrink only about 1.6-fold as factors, against the roughly ninefold saving the Python edition reports for the same columns as pandas categoricals. The difference is not a measurement error: R's `CHARSXP` cache already stores each distinct string once, which is most of what a factor would buy. In R the argument for factors is *semantics* — ordered levels, model contrasts, controlled levels in a plot — far more than memory.

The `.drop` demonstration is the trap. Counting two factor columns with `.drop = FALSE` returns every combination, including those that never occur: here a harmless 9 instead of 4, but with two columns of 300 levels it silently materialises 90,000 rows of structural zeros. The same applies to `table()`, which never drops unused levels; this course writes `count()` and keeps `.drop` at its default.

## 12. Writing it as a pipeline

Wrangling written as a sequence of reassignments (`df <- df[...]`, `df$x <- ...`) is hard to read and harder to debug: `df` means something different on every line and nothing can be inspected without re-running from the top. **Piping** — one expression, one verb per line — gives each step a visible input and output. R's native `|>` operator (R 4.1 and later) needs no package; `%>%` from magrittr offers extra features such as the `.` placeholder. Four verbs make a wrangling pipeline readable: `filter`, `mutate`, `select` and `summarise`; a fifth, `(\(d) ...)()`, inserts an arbitrary function.



In [ ]:
measurements <- load_data("rock1_chembl_raw.csv") |>
  filter(!is.na(canonical_smiles)) |>                                 # a structure is required
  filter(target_organism == "Homo sapiens") |>                        # human target only
  filter(is.na(data_validity_comment)) |>                             # drop ChEMBL's quality flags
  mutate(value_nM = to_nanomolar(value, units, type, relation)$nM,
         rel      = to_nanomolar(value, units, type, relation)$rel) |>
  filter(!is.na(value_nM), value_nM > 0, value_nM <= 1e8) |>          # possible concentrations
  inner_join(key_map, by = "molecule_chembl_id",                      # standardised parent (U6 recipe)
             relationship = "many-to-one") |>
  mutate(pActivity = 9 - log10(value_nM),
         censored  = rel != "=")
cat("measurement-level table:", nrow(measurements), "rows x", ncol(measurements), "columns\n")
cat("distinct compounds (InChIKey):", n_distinct(measurements$inchikey),
    "| exact:", sum(!measurements$censored),
    "| censored:", sum(measurements$censored), "\n")
print(as.data.frame(measurements |>
  select(molecule_chembl_id, standard_type, value_nM, rel, pActivity, censored) |>
  head(4) |> mutate(across(where(is.numeric), \(x) round(x, 3)))))



Five thousand and forty-five measurements survive, on 4,280 distinct structures, 3,328 of them exact and 1,717 censored. Read the pipeline as prose: *take the raw file, keep rows with a structure, keep the human target, drop the rows ChEMBL flagged, convert value and relation together, keep possible concentrations, attach the standardised parent, add the logarithmic potency and the censoring flag.* Each clause is one line and one idea, and deleting any line answers "what did that step cost?". The `inner_join` on `key_map` does the work of the Python edition's RDKit standardisation and heavy-atom filter, both already encoded in the frozen reference table; Unit 6 turns this pipeline into a protocol that *logs* what each line removed and why.

One inefficiency is left in deliberately: `to_nanomolar()` is called twice, once per output, where a single call whose result is unpacked would be cleaner and twice as fast. At 5,089 rows the cost is milliseconds; the next section is about knowing when that stops being true.

## 13. Performance: data.table, dplyr and the loop you should not write

Our table is small, so none of this matters yet; it will the first time you handle a million docking poses. Two rules: **let the vectorised primitive do the loop**, and **know when to reach for `data.table`**. `microbenchmark` reports the median of many repetitions rather than one wall-clock reading.



In [ ]:
s <- raw$standard_value[!is.na(raw$standard_value) & raw$standard_value > 0]
mb1 <- microbenchmark(
  `vectorised`  = 9 - log10(s),
  `sapply`      = sapply(s, function(v) 9 - log10(v)),
  `for loop`    = { o <- numeric(length(s)); for (i in seq_along(s)) o[i] <- 9 - log10(s[i]); o },
  `Map + unlist`= unlist(Map(function(v) 9 - log10(v), s)),
  times = 10L)
t1 <- summary(mb1, unit = "ms")[, c("expr", "median")]
t1$slowdown <- round(t1$median / min(t1$median), 1)
cat(sprintf("median milliseconds for %d values:\n", length(s)))
print(t1, row.names = FALSE, digits = 4)

dt <- as.data.table(raw)
mb2 <- microbenchmark(
  dplyr      = raw |> group_by(assay_chembl_id) |>
                 summarise(m = median(standard_value, na.rm = TRUE), .groups = "drop"),
  data.table = dt[, .(m = median(standard_value, na.rm = TRUE)), by = assay_chembl_id],
  times = 10L)
t2 <- summary(mb2, unit = "ms")[, c("expr", "median")]
cat("\nsame grouped median, two engines (median milliseconds):\n")
print(t2, row.names = FALSE, digits = 4)

d <- rbind(transform(t1[, c("expr", "median")], panel = "element-wise arithmetic"),
           transform(t2, panel = "grouped median, 276 groups"))
print(ggplot(d, aes(median, reorder(expr, median))) +
  geom_col(fill = PAL["blue"], width = 0.65) +
  geom_text(aes(label = sprintf("%.2f ms", median)), hjust = -0.1, size = 2.5) +
  scale_x_log10(expand = expansion(mult = c(0, 0.35))) +
  facet_wrap(~panel, scales = "free") +
  labs(x = "median milliseconds (log scale)", y = NULL, title = "Let the primitive do the loop"))



<p class="caption"><b>Figure 8.</b> Left panel: four implementations of pActivity = 9 − log<sub>10</sub>(value) on the same 5,062 positive values, on a logarithmic time axis. The vectorised expression is roughly ninety to a hundred times faster than the three that pay one R function call per element, and <code>sapply</code>, an explicit <code>for</code> loop and <code>Map</code> are within a factor of 1.2 of each other — the cost is the interpreted call, not the syntax you wrap it in. Right panel: the same grouped median over 276 assays, computed by dplyr and by <code>data.table</code>, which is several times faster here; the gap widens with the number of groups. Read the printed numbers rather than these words; the factors depend on the machine.</p>

Three further habits:

- **Read only what you need:** `fread(path, select = c(...))` on a 2 GB file is the difference between a working laptop and a swap death; `fread` also reads gzip directly when `R.utils` is installed, which is how the fingerprint files of Units 13 and 19 are loaded.
- **Never grow a data frame in a loop:** `rbind` inside a loop re-allocates the whole object every iteration and is quadratic; collect into a list and `bind_rows()` once.
- **Profile before optimising:** `microbenchmark` for an expression, `Rprof` or `profvis` for a script. For numerical work beyond vector arithmetic reach for a compiled primitive rather than writing your own — the advice the Python world gives about SciPy.<sup class="c"><a href="#r24">[24]</a></sup> The equivalents of this section's tools in the Python edition are `%timeit` in IPython<sup class="c"><a href="#r26">[26]</a></sup> and matplotlib<sup class="c"><a href="#r25">[25]</a></sup> for the figure.

## 14. The deliverable: from measurements to compounds

Everything so far prepares one transformation. The modelling table of Units 11–17 has **one row per compound**, and four decisions get us there, each written down rather than defaulted into:

1. **What is a compound?** The standardised parent structure keyed by InChIKey (Section 8), not the ChEMBL identifier.
2. **How are replicates combined?** The **median** of the exact values, insensitive to the single wild value a literature dataset always contains; the mean would be dragged by it.
3. **What about compounds with only censored measurements?** Kept, with the median of their censored values and a `censored = TRUE` flag. Dropping them would delete 1,516 mostly inactive compounds — the MNAR trap of Section 7.
4. **What provenance survives?** The number of measurements, assays and documents behind each row, and the replicate spread, which Unit 6 turns into an experimental noise floor.

<h4>Worked example by hand: fasudil</h4>

Fasudil is an approved ROCK inhibitor. Suppose its exact pActivity values were 6.52, 6.40, 6.46, 6.70, 6.22. Sorted: 6.22, 6.40, **6.46**, 6.52, 6.70 — the median is the third of five values, 6.46, and the range 6.70 − 6.22 = 0.48 log units, below the 1-log "wide spread" threshold. The chunk below runs the same aggregation on the real measurements.



In [ ]:
compounds <- measurements |>
  group_by(inchikey) |>
  mutate(use = if (any(!censored)) !censored else rep(TRUE, n())) |>   # exact values if any exist
  summarise(std_smiles      = first(std_smiles),
            chembl_ids      = paste(sort(unique(molecule_chembl_id)), collapse = "|"),
            pref_name       = if (all(is.na(pref_name))) NA_character_ else na.omit(pref_name)[1],
            pActivity_min   = min(pActivity[use]),     # min/max BEFORE the median: see below
            pActivity_max   = max(pActivity[use]),
            pActivity       = median(pActivity[use]),
            n_measurements  = n(),
            n_exact         = sum(!censored),
            n_censored      = sum(censored),
            n_assays        = n_distinct(assay_chembl_id),
            n_documents     = n_distinct(document_chembl_id),
            .groups = "drop") |>
  mutate(censored = n_exact == 0,
         pActivity_range = pActivity_max - pActivity_min,
         wide_spread = pActivity_range > 1)
cat("compound-level table:", nrow(compounds), "rows x", ncol(compounds), "columns\n")
cat("\nfasudil's row:\n")
print(as.data.frame(compounds |> filter(coalesce(pref_name, "") == "FASUDIL") |>
  select(pref_name, pActivity, pActivity_min, pActivity_max, n_measurements, n_exact,
         n_assays, n_documents) |> mutate(across(where(is.numeric), \(x) round(x, 3)))))
fas <- measurements |> filter(coalesce(pref_name, "") == "FASUDIL") |>
  select(standard_type, value_nM, rel, pActivity, assay_chembl_id)
cat("\nthe measurements behind it (first 8 of", nrow(fas), "):\n")
print(as.data.frame(fas |> head(8) |> mutate(across(where(is.numeric), \(x) round(x, 3)))))
cat(sprintf("median of its exact pActivity values: %.3f\n",
            median(fas$pActivity[fas$rel == "="])))



The order of the three `pActivity*` lines is not cosmetic. `summarise()` makes each new column visible to the expressions that follow it, so writing `pActivity = median(pActivity[use])` first would make the later `min(pActivity[use])` operate on the one-element summary rather than on the original vector, silently producing `NA`. In this document the mistake was caught by the invariant `all(pActivity_range >= 0)` three chunks later, which is precisely the service invariants are there to render.

The real fasudil row aggregates 18 measurements, all of them exact, from 16 assays and 15 documents into a median pActivity of 6.464 — an IC50 of about 340 nM, the value the literature quotes for fasudil against ROCK1. Its replicates run from 5.367 to 7.500, a range of 2.13 log units: the same molecule measured a hundredfold apart by different laboratories, which is why Section 6's assay effect matters and why the median rather than the mean is the aggregation of choice. The hand calculation used invented replicates to show the mechanics; the code applies the same mechanics to the 18 real ones.

Now the invariants. A transformation of this size deserves a test suite, and in a knitted document the test suite is a chunk of `stopifnot()` calls that must run silently.



In [ ]:
n_before <- nrow(measurements)
stopifnot(
  "the row key must be unique"                   = !any(duplicated(compounds$inchikey)),
  "no missing keys"                              = all(!is.na(compounds$inchikey)),
  "every measurement lands in exactly one row"    = sum(compounds$n_measurements) == n_before,
  "no missing target value"                      = all(!is.na(compounds$pActivity)),
  "no infinities (a zero concentration gives one)" = all(is.finite(compounds$pActivity)),
  "potency must be physically plausible"          = all(compounds$pActivity > 2 & compounds$pActivity < 11),
  "exact + censored = total"                     = all(compounds$n_exact + compounds$n_censored == compounds$n_measurements),
  "censored rows have no exact value"             = all(compounds$n_exact[compounds$censored] == 0),
  "uncensored rows have at least one"             = all(compounds$n_exact[!compounds$censored] > 0),
  "ranges are non-negative"                      = all(compounds$pActivity_range >= 0))
cat("all 10 invariants hold for", nrow(compounds), "compounds\n\n")
print(summary(compounds[c("pActivity", "pActivity_range", "n_measurements", "n_assays")]))
cat("\ncompounds with at least one exact value:", sum(!compounds$censored),
    "| censored-only:", sum(compounds$censored),
    "| wide spread (>1 log):", sum(compounds$wide_spread), "\n")



Ten invariants, ten lines, no output when they pass. Two deserve emphasis: `sum(n_measurements) == n_before` is a conservation law that would have caught the many-to-many explosion of Section 5 instantly, and `all(is.finite(pActivity))` guards against a concentration of exactly zero, whose logarithm is −∞. Writing the invariants as *named* arguments of `stopifnot()` turns each name into the error message. The summary shows the deliverable: 2,764 compounds carry at least one exact value, 1,516 are censored-only, the median compound has one measurement from one assay and the maximum is 24 measurements over 19 assays.

The final check is against an independent implementation: the course ships a reference curated table built by the laboratory's own protocol script, and our pipeline should reproduce it.



In [ ]:
ref <- curated
cat("our compounds:", nrow(compounds), "| reference compounds:", nrow(ref), "\n")
stopifnot(nrow(compounds) == nrow(ref))
stopifnot(setequal(compounds$inchikey, ref$inchikey))
merged <- compounds |>
  select(inchikey, pActivity_ours = pActivity, censored_ours = censored) |>
  inner_join(ref |> transmute(inchikey, pActivity_ref = pActivity,
                              censored_ref = censored == "True"),
             by = "inchikey", relationship = "one-to-one")
d <- abs(merged$pActivity_ours - merged$pActivity_ref)
cat("identical censoring flags:", all(merged$censored_ours == merged$censored_ref), "\n")
cat(sprintf("max |pActivity difference| vs reference: %.4f log units\n", max(d)))
cat("rows differing by more than 0.001:", sum(d > 1e-3), "\n")



Our pipeline reproduces the reference table: the same 4,280 structures, the same censoring flags, and potencies agreeing to 0.0005 log units — the residue of the reference file storing `pActivity` rounded to three decimals. This is the standard every dataset built in this laboratory is held to.<sup class="c"><a href="#r20">[20]</a></sup><sup class="c"><a href="#r21">[21]</a></sup> One R-specific trap is visible in the code: the reference CSV was written by pandas, so its boolean columns arrive as the *strings* `"True"` and `"False"`. Comparing a logical column with a character one returns `FALSE` for every row rather than raising, so the explicit `censored == "True"` is what makes the check mean anything.

Two last pictures: where the rows went, and how much the replicates disagree.



In [ ]:
has_struct <- !is.na(raw$canonical_smiles)
human <- has_struct & raw$target_organism == "Homo sapiens"
unflagged <- human & is.na(raw$data_validity_comment)
steps <- tibble::tibble(
  step = c("raw records", "has a structure", "human target", "no validity flag",
           "numeric, positive value", "standardised, 3-70 heavy atoms", "compounds (InChIKey)"),
  rows = c(nrow(raw), sum(has_struct), sum(human), sum(unflagged),
           sum(unflagged & !is.na(raw$value_nM) & raw$value_nM > 0 & raw$value_nM <= 1e8),
           nrow(measurements), nrow(compounds)))
steps <- steps |> mutate(step = factor(step, levels = rev(step)),
                         kind = c(rep("measurements", 6), "compounds"))
p1 <- ggplot(steps, aes(rows, step, fill = kind)) +
  geom_col(width = 0.7) +
  geom_text(aes(label = format(rows, big.mark = ",")), hjust = -0.12, size = 2.5) +
  scale_fill_manual(values = c(measurements = unname(PAL["blue"]),
                               compounds = unname(PAL["green"])), guide = "none") +
  scale_x_continuous(limits = c(0, 6100), expand = c(0, 0)) +
  labs(x = "rows remaining", y = NULL, title = "From 5,089 records to 4,280 compounds")
rep_spread <- compounds |> filter(n_exact > 1)
p2 <- ggplot(rep_spread, aes(pActivity_range)) +
  geom_histogram(binwidth = 0.1, boundary = 0, fill = PAL["purple"]) +
  geom_vline(xintercept = 1, colour = PAL["orange"], linetype = "dashed", linewidth = 0.5) +
  annotate("text", x = 1.08, y = Inf, vjust = 1.6, hjust = 0, size = 2.4,
           colour = unname(PAL["orange"]), label = "wide-spread\nthreshold") +
  labs(x = "max \u2212 min pActivity among a compound's exact replicates", y = "compounds",
       title = paste0("Replicate disagreement (n=", nrow(rep_spread), " compounds)"))
print(p1 | p2)
cat(sprintf("compounds with >1 exact measurement: %d | median range %.2f | 90th percentile %.2f log units | above 1 log: %d\n",
            nrow(rep_spread), median(rep_spread$pActivity_range),
            quantile(rep_spread$pActivity_range, 0.9), sum(rep_spread$pActivity_range > 1)))



<p class="caption"><b>Figure 9.</b> Left: pipeline attrition. Only 44 records are lost to filtering (3 without a structure, 13 carrying a validity flag, 26 with no number, 2 outside the heavy-atom window); the large final drop is aggregation, not loss — 5,045 measurements become 4,280 compounds. Right: for the 337 compounds with more than one exact measurement, the spread between the most and least potent replicate: the median is 0.00 and the 90th percentile 1.41, so most agree exactly while 55 of the 337 exceed one log unit — two laboratories disagreeing by more than tenfold about the same molecule. With censored-only compounds included, 60 rows of the final table carry the <code>wide_spread</code> flag. Unit 6 turns this tail into a quantitative ceiling on how accurate any model of these data can be — the effect quantified by Kramer <i>et al.</i>,<sup class="c"><a href="#r30">[30]</a></sup> Kalliokoski <i>et al.</i><sup class="c"><a href="#r29">[29]</a></sup> and Landrum and Riniker.<sup class="c"><a href="#r31">[31]</a></sup></p>

<div class="box bridge"><h4>Bridge</h4>
<p>This unit is the tidyverse half of the laboratory's data pipeline. <strong>Backwards:</strong> Unit 3 of this course obtained this table from the ChEMBL API and explains the identifiers and the SQL alternative (<a href="../U3_Getting_Data/index.html">Python edition</a>), and Unit 2 explains why the snapshot is frozen and hashed (<a href="../U2_Reproducibility/index.html">Python edition</a>). <strong>Forwards:</strong> Unit 5 (<a href="../U5_EDA/index.html">Exploratory data analysis and the grammar of graphics</a>) explores the compound table we just built, and Unit 6 (<a href="../U6_Data_Quality/index.html">Data quality, curation and bias</a>) turns the pipeline of Section 12 into a reason-coded protocol with a data card and an experimental noise floor; Units 11–12 consume the result as the modelling table (<a href="../U11_Supervised_Workflow/index.html">supervised workflow</a>, <a href="../U12_Validation/index.html">validation</a>), where the InChIKey de-duplication of Section 8 becomes the difference between an honest and a leaking split. The <a href="../index.html">R-edition index</a> lists every unit, and the <a href="../../Python/U4_Tidy_Data/index.html">Python edition of this unit</a> runs the same analysis with pandas. <strong>Sister courses:</strong> structure standardisation, salts and InChIKeys are developed in <a href="../../../Cheminformatics/U4_Standardisation/index.html">Unit 4</a> and <a href="../../../Cheminformatics/U7_Data_Curation/index.html">Unit 7</a> of the cheminformatics course and the descriptors in <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">Unit 5</a>; the consequences of a leaking split are the subject of <a href="../../../BGM411/R/S1_Model_Evaluation/index.html">Model evaluation</a> in the AI course (R edition); and the bioinformatics course applies the same discipline to sequences in <a href="../../../BGM324/01_Sequence_Alignment/index.html">Sequence alignment</a>.</p></div>

## 15. Summary

- **Tidy data is a contract**: one variable per column, one observation per row, one observational unit per table. `pivot_longer`, `pivot_wider`, `nest`/`unnest` and `separate_rows` move between shapes; every reshape has an invariant, so write it as a `stopifnot()`.
- **Decide the grain before cleaning.** The raw file has one row per measurement (5,089 rows, 4,312 compounds, 3,802 of them measured once), the modelling table one per compound (4,280). Conflating them is the commonest source of leakage in bioactivity modelling.
- **Joins fail three ways**: wrong cardinality (`relationship=`), silent non-matching (`anti_join`), undeclared row survival (the choice of verb). A careless join against a non-unique lookup turned 5,089 rows into 1,647,711 — and dplyr warned about it before we forced it through.
- **`summarise` and `mutate` are different semantics** on the same `group_by`. Centring with a grouped `mutate` showed 53.9% of the variance in potency is attributable to which assay measured it.
- **Missingness has a mechanism.** `pchembl_value` is missing in exactly the censored rows (MNAR); complete-case analysis was biased by 0.53 log units in our simulation, against 0.06 under MAR and none under MCAR.
- **"Duplicate" needs a definition**: exact-key (78 redundant rows), the database flag (279 rows, only 4 shared with the first set) and structural (9 InChIKeys covering 19 identifiers, including three deposits of ripasudil).
- **Harmonise units with a table, not cleverness**, convert value and relation in one function (550 relations flip when a pKi becomes a Ki), and check against an independent harmonisation — ours matched ChEMBL's to within 5.71 × 10<sup>−5</sup> log units on 5,062 rows, the residue being ChEMBL's rounding.
- **Keep censored measurements and flag them.** 1,727 of 5,089 are censored and 1,516 compounds have nothing else; they pile up at pActivity 5.00 (10 µM) and 4.52 (30 µM), so treating the spikes as potencies invents compounds.
- **Pipe the pipeline, let the primitive loop, finish with `stopifnot()`.** Ten invariants plus a comparison with the reference protocol are what let us claim the table is correct — the same 4,280 structures, the same flags, potencies within 0.0005 log units.

## 16. Self-assessment

<details><summary>1. A collaborator sends a spreadsheet with columns <code>compound</code>, <code>day1</code>, <code>day2</code>, <code>day3</code>. Which tidy rule is broken and what is the one-line fix?</summary>
<p>Rule 1: the headers <code>day1…day3</code> are <em>values</em> of a variable that has no column. Fix: <code>pivot_longer(df, -compound, names_to = "day", values_to = "value")</code>, after which "day" can be grouped, filtered and mapped to an aesthetic like any other variable.</p></details>

<details><summary>2. You join a 10,000-row measurement table to a 500-row "compound properties" table on <code>compound_id</code> and get 10,240 rows. What happened, and which single argument would have prevented it?</summary>
<p>The right-hand table is not unique on <code>compound_id</code>, so those measurements were duplicated. <code>relationship = "many-to-one"</code> raises an error instead of silently producing the extra 240 rows; even without it, dplyr emits a many-to-many warning that should be treated as an error.</p></details>

<details><summary>3. What is the difference between <code>group_by(...) |> summarise(m = median(x))</code> and <code>group_by(...) |> mutate(m = median(x))</code>?</summary>
<p><code>summarise</code> returns one row per group; <code>mutate</code> returns one value per <em>original</em> row — the group median broadcast back — so the result stays joined to the data. Summarise to report, grouped-mutate to normalise within groups. Remember to <code>ungroup()</code> afterwards.</p></details>

<details><summary>4. In our table, <code>pchembl_value</code> is missing in 1,759 rows. Classify the mechanism and justify it from the data.</summary>
<p>MNAR. The contingency table against <code>standard_relation</code> shows missingness is total for every censored row (1,410 <code>&gt;</code>, 312 <code>&lt;</code>, 5 <code>&lt;=</code>, 26 with no relation) and nearly absent for exact rows (6 of 3,336): the value is missing <em>because</em> the compound was too weak to measure, so complete-case analysis discards the inactives and biases mean potency upward.</p></details>

<details><summary>5. Why does <code>distinct(df, molecule_chembl_id)</code> not de-duplicate a chemical dataset?</summary>
<p>Because one molecule can carry several identifiers — free base, hydrochloride, hydrate, independent deposits. Ripasudil appears under three ChEMBL ids here, and nine InChIKeys absorb nineteen identifiers in all. De-duplicate on a canonical identifier of the standardised parent, such as the InChIKey.</p></details>

<details><summary>6. A paper reports "pIC50 &lt; 5". What is the IC50, and what is the relation symbol on the concentration scale?</summary>
<p>IC50 = 10<sup>−5</sup> M = 10,000 nM, and the relation becomes <code>&gt;</code>: the compound is <em>weaker</em> than 10 µM. The minus sign in −log<sub>10</sub> reverses the inequality. In our table 550 rows (518 pKi, 32 pIC50) undergo this flip.</p></details>

<details><summary>7. Why does this unit keep censored measurements instead of dropping them, and what would dropping them do to a regression model?</summary>
<p>They carry real information and amount to 1,727 of 5,089 measurements, including everything known about 1,516 compounds. Dropping them leaves actives only: the model never learns what an inactive looks like and prospective performance falls. Model them as censored, or carry the flag and exclude them explicitly from the regression target — and say so.</p></details>

<details><summary>8. What invariant would have caught a many-to-many join in the aggregation pipeline?</summary>
<p><code>stopifnot(sum(compounds$n_measurements) == nrow(measurements))</code> — the conservation law. A duplicated join inflates the per-group counts, so the sum exceeds the number of measurements that actually exist.</p></details>

<details><summary>9. In R, when is a factor a win, and what is the trap when counting on it?</summary>
<p>Chiefly for semantics: ordered levels, model contrasts, controlled ordering in a plot. The memory argument is weak in R — the four identifier columns here shrink only about 1.6-fold, because the <code>CHARSXP</code> cache already stores each distinct string once. The trap is <code>.drop = FALSE</code> in <code>count()</code> (and <code>table()</code>, which never drops): you then get every combination of levels, including those that never occur.</p></details>

<details><summary>10. You must aggregate replicate measurements of a compound. Why the median rather than the mean, and when would you prefer the mean?</summary>
<p>The median resists the single wild replicate literature data reliably contain: a mis-transcribed exponent moves a mean by a log unit and a median not at all. Fasudil's 18 replicates span 2.13 log units here. Prefer the mean for technical repeats of one experiment with symmetric error on the log scale, and report the standard deviation with it.</p></details>

<details><summary>11. Your cleaning script runs in 40 minutes on 2 million rows. Name the three things to check first.</summary>
<p>(i) An element-wise <code>sapply</code>, <code>Map</code> or <code>for</code> loop that should be vectorised — about a hundredfold above; (ii) <code>rbind</code> inside a loop, which is quadratic; (iii) reading every column when <code>fread(select =)</code> would read a tenth of the bytes. Then consider <code>data.table</code>. Profile first.</p></details>

<details><summary>12. Why does the final chunk compare our table with <code>rock1_chembl_curated.csv</code>, and what would a mismatch of two compounds mean?</summary>
<p>It is an independent-implementation check: two people wrote the same protocol in two languages, so structures, censoring flags and potencies must agree. A two-compound mismatch would mean one pipeline applied a filter the other did not — most likely the heavy-atom window or the standardisation step — and must be traced before either table is used, not averaged away.</p></details>

## Glossary

<dl class="gloss">
<dt>Anti-join</dt><dd>The rows of A whose key has no match in B; in dplyr the verb <code>anti_join()</code>, in pandas an idiom built from <code>indicator=True</code>.</dd>
<dt>Assay</dt><dd>One experimental protocol — enzyme preparation, substrate, detection method — with its own ChEMBL accession. One compound measured in two assays can differ by an order of magnitude.</dd>
<dt>Cardinality (of a join)</dt><dd>How many rows a key may match on each side; declared in dplyr with <code>relationship=</code>.</dd>
<dt>Censored measurement</dt><dd>A measurement reported as an inequality (<code>&gt; 10 µM</code>) because the experiment could not determine the value. Carries real information about inactivity.</dd>
<dt>Grain</dt><dd>What one row stands for: one measurement in the raw file, one compound in the modelling table.</dd>
<dt>IC50 / Ki / Kd</dt><dd>Concentrations summarising inhibition or binding: the concentration halving activity, the inhibition constant, the dissociation constant. Smaller is more potent.</dd>
<dt>InChIKey</dt><dd>A 27-character hash of a standardised structure; the practical key for de-duplicating chemical tables.</dd>
<dt>List column</dt><dd>A column whose cells are vectors or tables, made by <code>nest()</code> and flattened by <code>unnest()</code>.</dd>
<dt>MCAR / MAR / MNAR</dt><dd>Rubin's missingness mechanisms: completely at random (deletion unbiased), at random given observed variables (correctable), not at random (depends on the unobserved value; not correctable from the data alone).</dd>
<dt>pActivity</dt><dd>−log<sub>10</sub> of the activity in mol/L, i.e. 9 − log<sub>10</sub>(value in nM). One unit is tenfold.</dd>
<dt>Pipe</dt><dd>R's <code>|&gt;</code> (base, since 4.1) or <code>%&gt;%</code> (magrittr), which pass the left-hand value as the first argument of the right-hand call, so a wrangling sequence reads as one expression.</dd>
<dt>Split–apply–combine</dt><dd>The pattern behind <code>group_by</code>: partition, apply per group, combine — as a summary (<code>summarise</code>), a broadcast column (<code>mutate</code>) or a row filter (<code>filter</code>).</dd>
<dt>Tidy data</dt><dd>Each variable a column, each observation a row, each observational unit its own table.</dd>
<dt>Window function</dt><dd>Computed over a group but returning one value per row — <code>min_rank</code>, <code>row_number</code>, <code>lag</code>, <code>cumsum</code>. Order-dependent, so <code>arrange()</code> explicitly.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Wickham H. Tidy data. <i>Journal of Statistical Software</i> 2014;59(10):1–23. doi:<a href="https://doi.org/10.18637/jss.v059.i10">10.18637/jss.v059.i10</a></li>
<li id="r2">Broman KW, Woo KH. Data organization in spreadsheets. <i>The American Statistician</i> 2018;72(1):2–10. doi:<a href="https://doi.org/10.1080/00031305.2017.1375989">10.1080/00031305.2017.1375989</a></li>
<li id="r3">Zdrazil B, Felix E, Hunter F, <i>et al.</i> The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <i>Nucleic Acids Research</i> 2024;52(D1):D1180–D1192. doi:<a href="https://doi.org/10.1093/nar/gkad1004">10.1093/nar/gkad1004</a></li>
<li id="r4">Davies M, Nowotka M, Papadatos G, <i>et al.</i> ChEMBL web services: streamlining access to drug discovery data and utilities. <i>Nucleic Acids Research</i> 2015;43(W1):W612–W620. doi:<a href="https://doi.org/10.1093/nar/gkv352">10.1093/nar/gkv352</a></li>
<li id="r5">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <i>PLoS Computational Biology</i> 2013;9(10):e1003285. doi:<a href="https://doi.org/10.1371/journal.pcbi.1003285">10.1371/journal.pcbi.1003285</a></li>
<li id="r6">Cheng Y, Prusoff WH. Relationship between the inhibition constant (Ki) and the concentration of inhibitor which causes 50 per cent inhibition (I50) of an enzymatic reaction. <i>Biochemical Pharmacology</i> 1973;22(23):3099–3108. doi:<a href="https://doi.org/10.1016/0006-2952(73)90196-2">10.1016/0006-2952(73)90196-2</a></li>
<li id="r7">Wilson G, Bryan J, Cranston K, Kitzes J, Nederbragt L, Teal TK. Good enough practices in scientific computing. <i>PLoS Computational Biology</i> 2017;13(6):e1005510. doi:<a href="https://doi.org/10.1371/journal.pcbi.1005510">10.1371/journal.pcbi.1005510</a></li>
<li id="r8">Wickham H. The split-apply-combine strategy for data analysis. <i>Journal of Statistical Software</i> 2011;40(1):1–29. doi:<a href="https://doi.org/10.18637/jss.v040.i01">10.18637/jss.v040.i01</a></li>
<li id="r9">Rubin DB. Inference and missing data. <i>Biometrika</i> 1976;63(3):581–592. doi:<a href="https://doi.org/10.1093/biomet/63.3.581">10.1093/biomet/63.3.581</a></li>
<li id="r10">Sterne JAC, White IR, Carlin JB, <i>et al.</i> Multiple imputation for missing data in epidemiological and clinical research: potential and pitfalls. <i>BMJ</i> 2009;338:b2393. doi:<a href="https://doi.org/10.1136/bmj.b2393">10.1136/bmj.b2393</a></li>
<li id="r11">Weininger D. SMILES, a chemical language and information system. 1. Introduction to methodology and encoding rules. <i>Journal of Chemical Information and Computer Sciences</i> 1988;28(1):31–36. doi:<a href="https://doi.org/10.1021/ci00057a005">10.1021/ci00057a005</a></li>
<li id="r12">Bento AP, Hersey A, Félix E, <i>et al.</i> An open source chemical structure curation pipeline using RDKit. <i>Journal of Cheminformatics</i> 2020;12:51. doi:<a href="https://doi.org/10.1186/s13321-020-00456-1">10.1186/s13321-020-00456-1</a></li>
<li id="r13">Heller SR, McNaught A, Pletnev I, Stein S, Tchekhovskoi D. InChI, the IUPAC International Chemical Identifier. <i>Journal of Cheminformatics</i> 2015;7:23. doi:<a href="https://doi.org/10.1186/s13321-015-0068-4">10.1186/s13321-015-0068-4</a></li>
<li id="r14">Fourches D, Muratov E, Tropsha A. Trust, but verify: on the importance of chemical structure curation in cheminformatics and QSAR modeling research. <i>Journal of Chemical Information and Modeling</i> 2010;50(7):1189–1204. doi:<a href="https://doi.org/10.1021/ci100176x">10.1021/ci100176x</a></li>
<li id="r15">Fourches D, Muratov E, Tropsha A. Trust, but verify II: a practical guide to chemogenomics data curation. <i>Journal of Chemical Information and Modeling</i> 2016;56(7):1243–1252. doi:<a href="https://doi.org/10.1021/acs.jcim.6b00129">10.1021/acs.jcim.6b00129</a></li>
<li id="r16">Tiikkainen P, Bellis L, Light Y, Franke L. Estimating error rates in bioactivity databases. <i>Journal of Chemical Information and Modeling</i> 2013;53(10):2499–2505. doi:<a href="https://doi.org/10.1021/ci400099q">10.1021/ci400099q</a></li>
<li id="r17">Williams AJ, Ekins S. A quality alert and call for improved curation of public chemistry databases. <i>Drug Discovery Today</i> 2011;16(17–18):747–750. doi:<a href="https://doi.org/10.1016/j.drudis.2011.07.007">10.1016/j.drudis.2011.07.007</a></li>
<li id="r18">Tobin J. Estimation of relationships for limited dependent variables. <i>Econometrica</i> 1958;26(1):24–36. doi:<a href="https://doi.org/10.2307/1907382">10.2307/1907382</a></li>
<li id="r19">Helsel DR. Fabricating data: how substituting values for nondetects can ruin results, and what can be done about it. <i>Chemosphere</i> 2006;65(11):2434–2439. doi:<a href="https://doi.org/10.1016/j.chemosphere.2006.04.051">10.1016/j.chemosphere.2006.04.051</a></li>
<li id="r20">Wilkinson MD, Dumontier M, Aalbersberg IJ, <i>et al.</i> The FAIR Guiding Principles for scientific data management and stewardship. <i>Scientific Data</i> 2016;3:160018. doi:<a href="https://doi.org/10.1038/sdata.2016.18">10.1038/sdata.2016.18</a></li>
<li id="r21">Gebru T, Morgenstern J, Vecchione B, <i>et al.</i> Datasheets for datasets. <i>Communications of the ACM</i> 2021;64(12):86–92. doi:<a href="https://doi.org/10.1145/3458723">10.1145/3458723</a></li>
<li id="r22">McKinney W. Data structures for statistical computing in Python. <i>Proceedings of the 9th Python in Science Conference</i> 2010:56–61. doi:<a href="https://doi.org/10.25080/Majora-92bf1922-00a">10.25080/Majora-92bf1922-00a</a></li>
<li id="r23">Harris CR, Millman KJ, van der Walt SJ, <i>et al.</i> Array programming with NumPy. <i>Nature</i> 2020;585:357–362. doi:<a href="https://doi.org/10.1038/s41586-020-2649-2">10.1038/s41586-020-2649-2</a></li>
<li id="r24">Virtanen P, Gommers R, Oliphant TE, <i>et al.</i> SciPy 1.0: fundamental algorithms for scientific computing in Python. <i>Nature Methods</i> 2020;17:261–272. doi:<a href="https://doi.org/10.1038/s41592-019-0686-2">10.1038/s41592-019-0686-2</a></li>
<li id="r25">Hunter JD. Matplotlib: a 2D graphics environment. <i>Computing in Science &amp; Engineering</i> 2007;9(3):90–95. doi:<a href="https://doi.org/10.1109/MCSE.2007.55">10.1109/MCSE.2007.55</a></li>
<li id="r26">Perez F, Granger BE. IPython: a system for interactive scientific computing. <i>Computing in Science &amp; Engineering</i> 2007;9(3):21–29. doi:<a href="https://doi.org/10.1109/MCSE.2007.53">10.1109/MCSE.2007.53</a></li>
<li id="r27">Wickham H, Averick M, Bryan J, <i>et al.</i> Welcome to the Tidyverse. <i>Journal of Open Source Software</i> 2019;4(43):1686. doi:<a href="https://doi.org/10.21105/joss.01686">10.21105/joss.01686</a></li>
<li id="r28">Codd EF. A relational model of data for large shared data banks. <i>Communications of the ACM</i> 1970;13(6):377–387. doi:<a href="https://doi.org/10.1145/362384.362685">10.1145/362384.362685</a></li>
<li id="r29">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data — a statistical analysis. <i>PLoS ONE</i> 2013;8(4):e61007. doi:<a href="https://doi.org/10.1371/journal.pone.0061007">10.1371/journal.pone.0061007</a></li>
<li id="r30">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <i>Journal of Medicinal Chemistry</i> 2012;55(11):5165–5173. doi:<a href="https://doi.org/10.1021/jm300131x">10.1021/jm300131x</a></li>
<li id="r31">Landrum GA, Riniker S. Combining IC50 or Ki values from different sources is a source of significant noise. <i>Journal of Chemical Information and Modeling</i> 2024;64(5):1560–1567. doi:<a href="https://doi.org/10.1021/acs.jcim.4c00049">10.1021/acs.jcim.4c00049</a></li>
<li id="r32">Mendez D, Gaulton A, Bento AP, <i>et al.</i> ChEMBL: towards direct deposition of bioassay data. <i>Nucleic Acids Research</i> 2019;47(D1):D930–D940. doi:<a href="https://doi.org/10.1093/nar/gky1075">10.1093/nar/gky1075</a></li>
<li id="r33">Chambers J, Davies M, Gaulton A, <i>et al.</i> UniChem: a unified chemical structure cross-referencing and identifier tracking system. <i>Journal of Cheminformatics</i> 2013;5:3. doi:<a href="https://doi.org/10.1186/1758-2946-5-3">10.1186/1758-2946-5-3</a></li>
<li id="r34">Young D, Martin T, Venkatapathy R, Harten P. Are the chemical structures in your QSAR correct? <i>QSAR &amp; Combinatorial Science</i> 2008;27(11–12):1337–1345. doi:<a href="https://doi.org/10.1002/qsar.200810084">10.1002/qsar.200810084</a></li>
<li id="r35">Grolemund G, Wickham H. Dates and times made easy with lubridate. <i>Journal of Statistical Software</i> 2011;40(3):1–25. doi:<a href="https://doi.org/10.18637/jss.v040.i03">10.18637/jss.v040.i03</a></li>
<li id="r36">Wickham H. A layered grammar of graphics. <i>Journal of Computational and Graphical Statistics</i> 2010;19(1):3–28. doi:<a href="https://doi.org/10.1198/jcgs.2009.07098">10.1198/jcgs.2009.07098</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <i>R for Data Science</i>, 2nd ed. O'Reilly, 2023 — chapters 3–6, 19–20; free online. The canonical companion to this unit.</li>
<li>Wickham H. <i>Advanced R</i>, 2nd ed. Chapman &amp; Hall/CRC, 2019 — vectors, copy-on-modify and why the loop is slow.</li>
<li>Little RJA, Rubin DB. <i>Statistical Analysis with Missing Data</i>, 3rd ed. Wiley, 2019 — Section 7's mechanisms.</li>
<li>McKinney W. <i>Python for Data Analysis</i>, 3rd ed. O'Reilly, 2022 — for readers moving between the two editions of this course.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>dplyr vignettes: "Introduction to dplyr", "Two-table verbs", "Grouped data", "Window functions".</li>
<li>tidyr vignettes: "Pivoting", "Rectangling", "Nested data".</li>
<li><code>data.table</code> introduction vignette and the <i>data.table vs dplyr</i> comparison on its wiki; the <code>microbenchmark</code> reference manual.</li>
<li>ggplot2 book (Wickham, 3rd ed., free online)<sup class="c"><a href="#r36">[36]</a></sup> — the grammar used for every figure here and the subject of Unit 5.</li>
<li>The Turing Way, <i>Guide for Reproducible Research</i>; ChEMBL web-services documentation and schema diagram (EMBL-EBI).</li>
</ul>
